# 📋 [K09] 부대 정보 검색 시스템 - 부대 생활 안내 질의응답 도우미 만들기
### 학생 실습 워크북 & 프롬프트 설계 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K09/K09_student.ipynb)

---

## 📌 1. 프로젝트 개요 및 배경

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 병영생활·상담 (Level 2) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 15.0시간) |
| **실습 유형** | 생성형 AI 프롬프트 설계, 가상 데이터 분석, 대조 점검 워크북 |

### 📖 프로젝트 목표
새로 전입한 장병은 의무실 운영 시간, 사이버지식정보방 이용 수칙, 도서관 대출 방법 등 부대 생활에 필요한 정보를 어디서 어떻게 찾아야 하는지 막막해합니다. 이 프로젝트에서는 가상 부대(한빛대대)의 생활 안내문과 질문 카드를 활용하여, 장병들이 자주 묻는 질문에 정확하게 답변하는 생성형 AI 기반 질의응답 도우미를 구현합니다.

> **⚠️ [교육용 가상 자료 원칙]**
> - 본 실습의 모든 부대 명칭, 인명, 시설 및 규정은 **교육 목적으로 제작된 가상 데이터셋(가상 한빛대대)**입니다.
> - 실제 군사 기밀이나 보안 정보는 외부 AI 서비스에 일체 입력하지 않습니다.

---

> **💡 [실습 진행 안내]**
> - 본 실습은 파이썬 코딩 지식을 요구하지 않으며, 공개 생성형 AI(ChatGPT, Claude 등)와의 상호작용 및 팩트체크 검증을 기록하는 포트폴리오입니다.
> - 각 미션의 코드 셀 따옴표(`""" ... """`) 사이에 직접 설계한 프롬프트와 검증 결과를 입력하고 실행(`Shift + Enter`)하면 실습 리포트가 완성됩니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k09_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K09", "../uploads/K09"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k09_m01~m10) 준비 완료!")



---

## 🎯 미션 01. 자주 묻는 부대 생활 정보 파악하기
**소요시간**: 약 1.5시간

![미션 1 부대 생활 질문 분류 및 신병 우선순위 분석 다이어그램](/uploads/common/K09_m01_question_taxonomy.png)

### 📋 미션 개요 및 배경
신규 전입 장병이 부대 적응 과정에서 겪는 정보 탐색의 비효율과 반복적인 구두 질문으로 인한 행정 소요를 분석합니다. 실습자료의 질문 카드 20개를 시설·시간·절차·담당 부서 기준으로 분류하고, 전입 첫 주에 반드시 필요한 핵심 질문 5개를 선별합니다. 기존 정보 탐색 방식의 한계를 진단하여 생성형 AI 기반 안내 도우미가 해결해야 할 핵심 과업을 명확히 정의합니다.

### 🎯 미션 학습목표
실습자료의 질문 카드 20개를 시설·시간·절차·담당 부서로 분류하고, 전입 첫 주에 가장 필요한 핵심 질문 5개를 선별하여 현재 정보 탐색 방식의 불편점을 분석합니다. **이를 통해 사용자 관점에서 현장 문제를 정의하고, AI 도입이 필요한 우선순위 요구사항을 분석·선정하는 문제 도출 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 질문 카드 20개를 시설·시간·절차·담당 부서로 분류한다.
#### 이 단계의 역할
가상 한빛대대 장병들의 부대 생활 질문 카드 20개를 시설 위치, 운영 시간, 신청 절차, 담당 부서의 4대 카테고리로 체계적으로 분류합니다.

#### 해야 할 일

<li>실습 데이터(JSON)에 수록된 20개의 가상 질문 카드 원문을 꼼꼼히 확인합니다.

</li><li>각 질문의 핵심 키워드를 파악하여 [시설 위치, 운영 시간, 신청 절차, 담당 부서] 카테고리 분류표에 배치합니다.

</li><li>AI에게 복합적이거나 분류가 애매한 질문에 대한 최적의 카테고리 분류 기준을 자문하고 조율합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[질문 카테고리 분류 질의]:** "너는 군 행정 및 병영생활 상담 전문가야. 다음 장병들의 질문 20개를 [시설 위치, 운영 시간, 신청 절차, 담당 부서] 4개 카테고리로 분류하고, 분류 기준이 모호한 질문은 그 판단 근거를 간단히 적어줘.
[질문 카드 목록]: {질문 카드 20개 원문}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **누락 없는 전수 분류:** 20개 질문 카드가 중복이나 누락 없이 모두 분류표에 포함되었는가?
- **카테고리 타당성:** 단순 위치 문의(예: 사이버지식정보방 위치)와 운영 시간 및 행정 절차가 명확히 구분되었는가?

#### 핵심 확인 지표
<blockquote>20개 질문 카드 4대 영역 분류 완료표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>질문 카드 20개 분류표 완성본

</li>

#### 🔹 Step 2. 전입 첫 주 우선 질문 5개를 고른다.
#### 이 단계의 역할
20개 질문 중 신규 전입 장병이 자대 배치 첫 주 적응 과정에서 가장 시급하게 필요로 하는 우선순위 Top 5 질문을 선별합니다.

#### 해야 할 일

<li>전입 1~7일 차 장병의 필수 생활 동선(생활관, 식당, 의무실, PX, 복지시설)을 시뮬레이션합니다.

</li><li>질문 카드 20개 중 미확인 시 생활 불편이나 규정 위반이 발생할 수 있는 5개 핵심 질문을 선정합니다.

</li><li>AI에게 신병 관점에서의 질문 시급도 및 중요도 평가를 요청하고 본인의 선별안과 비교 검토합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[신병 우선 질문 선별 질의]:** "신병이 자대에 전입 온 첫 주에 겪는 주요 애로사항을 고려할 때, 다음 20개 질문 중 가장 시급하고 중요한 우선 질문 5개를 추천하고 선별 이유를 설명해 줘.
[질문 목록]: {분류된 20개 질문}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **우선순위 적절성:** 의무실 이용 시간, 식사 시간, 필수 행정 절차 등 긴급도 높은 질문이 포함되었는가?
- **선별 사유 명확성:** 각 질문이 전입 첫 주에 왜 우선되어야 하는지 타당한 근거가 제시되었는가?

#### 핵심 확인 지표
<blockquote>전입 첫 주 우선 질문 5개 및 선정 사유서

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>우선 질문 5개 목록 및 선정 근거서

</li>

#### 🔹 Step 3. 현재 정보 탐색 방식과 불편한 점을 적는다.
#### 이 단계의 역할
선임병 구두 질의, 게시판 공지 확인 등 기존 정보 탐색 경로의 한계와 병영 내 소통 단절 문제를 구체적으로 분석합니다.

#### 해야 할 일

<li>선임병 질문, 행정반 방문, 게시판 육안 확인 등 기존 방식의 소요 시간과 심리적 부담을 정리합니다.

</li><li>구두 전파 과정에서의 정보 왜곡, 담당자 부재 시 확인 지연 등 구체적인 병목 현상을 3가지 이상 도출합니다.

</li><li>AI 기반 부대 생활 안내 도우미 도입 시 기대되는 개선 효과(시간 절약, 익명성, 정확도)를 대조표로 작성합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[기존 탐색 방식 한계 분석 질의]:** "군 부대에서 신병이 선임병이나 간부에게 매번 구두로 질문할 때 겪는 심리적 부담과 정보 왜곡 문제를 분석해 줘. 이를 생성형 AI 도우미로 대체했을 때 기대되는 실질적인 업무/생활 개선 효과 3가지를 도출해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **현장 애로사항 반영:** 군 생활 현장에서 실제로 발생하는 현실적인 소통 불편 사항이 구체적으로 기술되었는가?
- **AI 도입 당위성 연결:** 기존 방식의 문제점이 향후 AI 안내 도우미 구축 목적과 논리적으로 직결되는가?

#### 핵심 확인 지표
<blockquote>정보 탐색 방식 비교 분석표 및 문제 진단서

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>현재 정보 탐색 방식과 불편점 정리서

</li>

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할
기본 실습 완료 후 실제 자대 환경을 반영한 추가 질문 2건을 직접 고안하여 분류 체계를 확장합니다.

#### 해야 할 일
실제 전입 시 가장 궁금했던 현실적 질문 2개를 직접 추가하고 카테고리를 분류해 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[자대 맞춤형 심화 질문 개발 질의]:** "부대 생활 안내 시스템에 다음 2개 질문을 추가하려 해. [질문 1: 주말 생활관 세탁기/건조기 이용 순번 규칙, 질문 2: 전투휴무일 종교시설 및 복지시설 운영 시간]. 이 질문들이 신병 적응에 기여하는 바와 적합한 카테고리를 평가해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **군사보안 준수:** 실제 부대 고유 명칭, 작전 계획, 간부 개인정보 등 비인가 정보가 일체 포함되지 않았는가?
- **질문 실효성:** 추가된 질문이 병영 생활 편의와 규정 준수에 실질적으로 기여하는가?

#### 핵심 확인 지표
<blockquote>심화 질문 2건 추가 및 타당성 분석 메모

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>심화 도전 과제 수행 결과 및 분석 메모

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M01] 자주 묻는 부대 생활 정보 파악하기 실습 가이드 및 작성 서식

## 1. 부대 생활 질문 20개 분류표
- 질문 카드 20건을 시설 위치, 운영 시간, 신청 절차, 담당 부서 4대 영역으로 분류합니다.

| 카테고리 | 질문 번호 (ID) | 질문 내용 요약 | 분류 사유 |
| :--- | :--- | :--- | :--- |
| **시설 위치** | Q07, Q16, ... | 영내 분실물 보관소 및 흡연장 위치 문의 | 시설의 공간적 위치 확인 목적 |
| **운영 시간** | Q01, Q02, Q03, Q04... | 의무실, 사지방, 체단실, PX 운영 시간 | 이용 가능한 일과 및 주말 시간 확인 |
| **신청 절차** | Q05, Q08, Q09, Q15... | 도서 대출, 휴가 서류, 이발실 예약, 상담 신청 | 서류 제출 및 사전 승인 절차 수반 |
| **담당 부서** | Q11, Q12, Q14... | 자격증 지원 담당과, 우편물 수령처 | 문의 및 처리를 담당하는 주관 부서 확인 |

---

## 2. 전입 첫 주 우선 질문 5선 (Top 5)
- 신병이 자대 적응 과정에서 가장 시급하게 알아야 할 질문 5개를 선별합니다.

1. **우선 질문 1 (Q01):** 의무실 일반 진료 시간 및 응급 접수 방법
   - **선정 사유:** 건강 및 응급 상황 대응과 직결되는 최우선 필수 복지 정보
2. **우선 질문 2 (Q04):** 복지매장(PX) 운영 시간 및 이용 규칙
   - **선정 사유:** 전입 초기 필수 세면도구 및 생필품 조달을 위한 필수 시설
3. **우선 질문 3 (Q02):** 사이버지식정보방(사지방) 이용 가능 시간
   - **선정 사유:** 일과 후 가족/지인 연락 및 자격증 준비를 위한 핵심 여가 시설
4. **우선 질문 4 (Q06):** 생활관 세탁기/건조기 이용 순번 및 시간
   - **선정 사유:** 매일 발생하는 피복 세탁 및 생활관 마찰 방지를 위한 기본 수칙
5. **우선 질문 5 (Q08):** 휴가/외출 신청 서류 제출처 및 기한
   - **선정 사유:** 첫 신병 위로휴가 준비 및 군 행정 절차 숙지 필수 항목

---

## 3. 현재 정보 탐색 방식과 불편한 점 (현장 문제 분석)
- **현재 탐색 경로:** 선임병 구두 질의(70%), 행정반 간부 직접 문의(20%), 게시판 육안 확인(10%)
- **주요 불편 사항:**
  1. *심리적 부담:* 전입 초기 맞선임이나 분대장에게 사소한 질문을 거듭하기 눈치 보임.
  2. *정보 왜곡 위험:* 구두 전파 특성상 선임의 기억 오류로 잘못된 시간이나 규정이 전달됨.
  3. *확인 지연:* 일과 중 담당 간부 부재 시 당일 확인이 불가능하여 행정 서류 제출 기한을 놓침.
- **AI 안내 도우미 도입 기대효과:**
  - 24시간 언제든 눈치 보지 않고 표준화된 규정을 3초 내 확인 가능
  - 공식 원문 데이터 기반 답변으로 정보 왜곡 원천 차단

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **가상 한빛대대 병영생활 질문 카드 20선 (JSON/MD):** 신규 전입 장병들이 실제로 겪는 다빈도 질의 20건 원문 데이터셋
- **부대 생활 질문 분류 및 우선순위 선정 서식 (MD/XLSX):** 4대 카테고리(시설, 시간, 절차, 부서) 분류 및 Top 5 우선순위 도출 시트
- **[학습 가이드] 사용자 중심 문제 정의 및 페르소나 분석법:** 신병의 시각에서 병목 현상을 진단하고 AI 도입 우선순위를 분석하는 실무 가이드

**실습 팁:** 20개 질문을 나눌 때는 장병들이 '어디로 가야 하는가(시설)', '언제까지 가능한가(시간)', '어떤 서류를 내는가(절차)'를 기준으로 분리하면 명확합니다.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] 문제 정의 — 자주 묻는 질문 20개를 체계적으로 분류하고 AI 안내 도우미 도입 필요성과 우선순위를 도출했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>20개 질문 카드 4대 영역 분류표 완성

</li><li>신병 전입 첫 주 우선 질문 5선 및 선정 사유 타당성 확보

</li><li>기존 정보 탐색 경로의 한계 및 AI 도우미 도입 기대효과 정리

</li><li>교육용 가상 데이터셋 활용 및 군사보안 수칙 엄수

</li>

#### 채점자 판정 메모
<blockquote>단순 질문 나열에 그치지 않고, 신병의 실제 애로사항(눈치, 정보 왜곡 등)을 정확히 진단하고 AI 도입 목표와 연결했는지 확인한다.

</blockquote>





In [ ]:
# [미션 01. 자주 묻는 부대 생활 정보 파악하기] 학생 실습 프롬프트 및 결과 기록란
mission_01_prompt = """
# [TODO: 미션 01] 질문 카드 20개를 4대 카테고리로 분류하고 전입 첫 주 우선순위 Top 5를 도출하는 프롬프트를 작성하세요.
# 힌트: load_dataset(1)로 20개 질문 카드를 확인한 후, AI에게 카테고리 분류 기준과 우선순위 선별을 지시합니다.
# (여기에 작성한 프롬프트를 입력하세요)
"""

mission_01_result = """
### 📋 [미션 01 실습 결과]
## 1. 부대 생활 질문 20개 분류표
| 카테고리 | 질문 번호 (ID) | 질문 내용 요약 | 분류 사유 |
| :--- | :--- | :--- | :--- |
| **시설 위치** | | | |
| **운영 시간** | | | |
| **신청 절차** | | | |
| **담당 부서** | | | |

## 2. 전입 첫 주 우선 질문 5선 (Top 5)
1. 
2. 
3. 
4. 
5. 

## 3. 현재 정보 탐색 방식의 한계와 AI 도우미 도입 효과
- **현장 문제점**: 
- **AI 도입 효과**:
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 01. 자주 묻는 부대 생활 정보 파악하기", mission_01_prompt, mission_01_result)



---

## 🎯 미션 02. 부대 안내 자료 정리하기
**소요시간**: 약 1.5시간

![미션 2 부대 안내 줄글 데이터의 5대 표준 표 구조화 다이어그램](/uploads/common/K09_m02_data_structuring.png)

### 📋 미션 개요 및 배경
생성형 AI가 거짓 정보를 생성하지 않도록 신뢰할 수 있는 부대 생활 참조 데이터를 체계적으로 구조화합니다. 가상 한빛대대 시설 안내 12건을 시설명·위치·운영 시간·연락처·제공 업무 항목으로 표준화된 표로 정리하고, 소독 시간이나 야간 당직 등 예외 운영 조건을 명시합니다. AI가 정밀하게 참조할 수 있는 비정형 텍스트의 표 구조화(데이터 정제) 역량을 체득합니다.

### 🎯 미션 학습목표
가상 한빛대대 시설 안내 12건을 시설명·위치·운영 시간·연락처·제공 업무 항목의 표로 정리하고, 소독 시간 및 야간 당직 등 운영 예외 조건을 명확히 표시합니다. **이를 통해 비정형 텍스트 안내문을 AI가 정확히 검색하고 파싱할 수 있는 표준 데이터 포맷으로 전처리·구조화하는 데이터 정제 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 시설 12건을 자료표 항목으로 정리한다.
#### 이 단계의 역할
가상 한빛대대의 줄글형 시설 안내문 12건을 AI가 정확히 탐색할 수 있는 표준 표(Table) 구조로 정제합니다.

#### 해야 할 일

<li>가상 한빛대대 시설 안내문 12건 원문을 꼼꼼히 확인합니다.

</li><li>[시설명, 위치, 운영 시간, 문의처, 제공 업무]의 5대 필수 컬럼으로 구성된 표 서식을 준비합니다.

</li><li>AI에게 줄글 텍스트를 마크다운 표로 변환하도록 요청하고, 변환된 표의 필드를 원문과 대조합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[시설 안내 데이터 표 구조화 질의]:** "너는 데이터 구조화 전문가야. 아래 부대 시설 안내 줄글 12건을 [시설명, 위치, 운영 시간, 문의처, 제공 업무] 컬럼을 가진 마크다운 표(Table) 형식으로 일목요연하게 변환해 줘. 빠진 내용이 없어야 해.
[안내문 12건]: {가상 시설 안내 텍스트}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **12건 전수 포함:** 의무실, 사이버지식정보방, 체력단련실, PX 등 12개 시설이 누락 없이 표에 기재되었는가?
- **컬럼 규격화:** 5대 필수 항목이 공통 서식으로 빈틈없이 채워졌는가?

#### 핵심 확인 지표
<blockquote>12건 부대 시설 표준 자료표 초안

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>부대 생활 안내 자료표(가상) 초안

</li>

#### 🔹 Step 2. 운영 시간 예외(소독·야간 당직)를 표시한다.
#### 이 단계의 역할
정규 운영 시간 외에 점심 소독 시간(Break time), 주말/공휴일 단축, 야간 당직 진료 등 특이 조건을 별도 식별하여 표시합니다.

#### 해야 할 일

<li>12개 시설의 원문에서 '단, ~는 제외', '소독 시간', '당직 접수' 등 예외 조항을 색출합니다.

</li><li>표의 '운영 시간' 컬럼에 정규 시간과 예외 조건을 괄호 또는 별도 표기로 명확히 구분하여 기재합니다.

</li><li>AI에게 조건부 예외 운영이 존재하는 시설 목록을 추출하도록 질의하여 누락을 크로스체크합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[운영 예외 조건 추출 질의]:** "정리된 시설 안내표에서 점심시간 소독, 정기 방역, 야간 당직, 주말 단축 등 '정규 시간 외 예외 조건'이 존재하는 시설들만 따로 추출하여 예외 내용과 이용 시 주의사항을 정리해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **예외 조건 정확도:** 의무실 소독 시간(12:00~13:30)이나 당직 접수 조건 등이 원문과 일치하는가?
- **시각적 가독성:** 정규 운영 시간과 예외 조건이 혼동되지 않게 명확히 구분 표기되었는가?

#### 핵심 확인 지표
<blockquote>예외 운영 조건이 보강된 부대 생활 안내 자료표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>부대 생활 안내 자료표(가상) 완성본

</li>

#### 🔹 Step 3. 심화 도전 과제 및 최적화
#### 이 단계의 역할
안내 자료표에 '자주 묻는 질문(FAQ)' 열을 추가하여 AI가 장병들의 질문 의도와 답변을 즉각 매핑할 수 있도록 데이터셋을 확장합니다.

#### 해야 할 일
자료표에 각 시설별 '자주 묻는 질문' 칸을 추가하여 데이터를 보강해 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[시설별 FAQ 확장 질의]:** "아래 12개 시설 안내표에 각 시설마다 장병들이 실제로 물어볼 만한 대표 질문 2개씩을 추가하여 '자주 묻는 질문(FAQ)' 컬럼을 확장한 마크다운 표로 만들어 줘.
[자료표]: {완성된 자료표}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **구어체 질문 반영:** 실제 장병들이 사용하는 자연스러운 질문 형태가 반영되었는가?
- **데이터 일관성:** 확장된 질문들이 기존 시설 정보 범위 내에서 답변 가능한가?

#### 핵심 확인 지표
<blockquote>FAQ 열이 확장된 고도화 부대 안내 자료표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>심화 확장 부대 생활 안내 자료표

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M02] 부대 안내 자료 정리하기 실습 가이드 및 작성 서식

## 1. 부대 생활 안내 자료표 (가상 한빛대대 표준 데이터셋)
- 가상 한빛대대 시설 12건의 줄글 원문을 바탕으로 AI가 검색하기 용이한 5대 표준 컬럼 표로 구조화합니다.

| 연번 | 시설명 | 위치 | 정규 운영 시간 | 예외 운영 조건 (소독/휴무/단서) | 문의 연락처 | 주요 제공 업무 |
| :---: | :--- | :--- | :--- | :--- | :--- | :--- |
| 1 | 대대 의무실 | 본관 1층 서편 | 평일 09:00~17:30 | 12:00~13:30 소독(진료 불가), 야간/휴일 당직 접수 | 내선 103 | 일반 진료, 약 조제, 물리치료 |
| 2 | 사이버지식정보방 | 복지회관 2층 201호 | 평일 17:30~21:00, 주말 09:00~21:00 | 매주 화요일 20:30~21:00 정기 점검 | 내선 205 | 인터넷 PC, 원격강좌, 자격증 접수 |
| 3 | 체력단련실 | 본관 지하 1층 | 평일 06:30~21:00, 주말 08:00~21:00 | 평일 일과시간 개인 이용 불가, 매주 수 14~16시 방역 | 내선 104 | 웨이트 트레이닝, 유산소, 샤워장 |
| 4 | 복지매장(PX) | 복지회관 1층 | 평일 12:00~19:30, 토 10:00~18:00 | 매주 일요일 정기 휴무, 평일 14~15시 재고조사 | 내선 201 | 생필품, 식음료, 택배 접수 |
| 5 | 병영도서관 | 복지회관 2층 204호 | 화~일 13:00~21:00 | 매주 월요일 정기 휴관, 1인 3권 14일 대출 | 내선 208 | 도서 대출 및 열람, 신문 구독 |
| 6 | 병영 세탁실 | 각 생활관 층별 중앙 | 매일 06:00~21:30 | 점호 준비(21:30 이후) 작동 금지, 분대별 요일제 | 각 중대 행정반 | 세탁기, 건조기, 다림질 |
| 7 | 부대 이발실 | 복지회관 지하 1층 | 화·목·토 13:30~17:30 | 사전 중대별 예약제 (선착순 방문 불가) | 내선 203 | 군 표준 두발 정리 |
| 8 | 영내 종교관 | 연병장 우측 종교시설동 | 수 19:00~20:30, 일 09:30~12:00 | 평일 개인 방문 시 군종병 사전 승인 필수 | 내선 109 | 주말 정기 예배/법회/미사, 종교 상담 |
| 9 | 우편물보관소 | 본관 1층 당직사령실 옆 | 평일 17:30~18:30 (수령 시간) | 사제 택배는 위병소 보안검색 후 익일 수령 | 내선 101 | 등기, 소포, 택배 수령 및 발송 |
| 10 | 병영생활상담실 | 본관 2층 동편 | 평일 09:00~18:00 | 100% 비밀 보장, 24시간 긴급 핫라인 운영 | 내선 119 | 1:1 고충 상담, 심리검사 |
| 11 | 가족 면회실 | 위병소 옆 단독 건물 | 토·일·공휴일 09:00~17:00 | 평일 면회 불가, 최소 3일 전 행정반 신청 | 내선 102 | 면회, 배달음식 취식, 휴게 공간 |
| 12 | 한빛 병영식당 | 본관 뒤편 급양시설동 | 조식 07~08, 중식 12~13, 석식 17:30~18:30 | 매주 토요일 브런치데이(09:00~10:30) 운영 | 내선 105 | 정규 급식, 식단표 공시, 브런치 |

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **가상 한빛대대 시설 안내문 12건 원문 (MD):** 의무실, PX, 체력단련실, 사지방 등 12개 부대 핵심 시설 줄글 안내문
- **표준 부대 생활 안내 자료표 서식 (Markdown Table):** 시설명, 위치, 정규 운영 시간, 연락처, 업무 5대 표준 컬럼 템플릿
- **[학습 가이드] 비정형 텍스트의 표(Table) 구조화 및 데이터 정제 기법:** 생성형 AI가 오류 없이 파싱할 수 있도록 예외 조항(소독, 당직)을 명시하는 데이터 전처리 가이드

**실습 팁:** '점심시간 소독(12:00~13:30)'이나 '야간 당직 진료' 같은 예외 조건을 누락하면 AI가 진료 불가 시간에 장병을 헛걸음하게 만들 수 있으므로 반드시 표기하세요.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] 정보 구조화 — AI 검색에 최적화된 5대 표준 컬럼으로 안내 자료를 정리하고 운영 예외 조건을 누락 없이 표시했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>12개 부대 시설 표준 안내 자료표 완성도 충족

</li><li>점심 소독 시간, 야간 당직, 요일제 등 운영 예외 조건 100% 반영

</li><li>가상 원문 데이터와 수치(시간, 위치, 내선번호) 1:1 일치

</li>

#### 채점자 판정 메모
<blockquote>의무실 점심 소독 시간(12:00~13:30)과 체력단련실 방역 시간 등 장병 헛걸음을 유발할 수 있는 예외 조건이 빠짐없이 적혔는지 육안 대조한다.

</blockquote>





In [ ]:
# [미션 02. 부대 안내 자료 정리하기] 학생 실습 프롬프트 및 결과 기록란
mission_02_prompt = """
# [TODO: 미션 02] 줄글 형태의 시설 안내문 12건을 5대 표준 컬럼 표로 구조화하는 프롬프트를 작성하세요.
# 힌트: load_dataset(2)의 facilities 항목을 확인하고, 예외 운영 조건(소독, 점검, 야간)이 누락되지 않도록 지시합니다.
# (여기에 작성한 프롬프트를 입력하세요)
"""

mission_02_result = """
### 📋 [미션 02 실습 결과]
## 1. 부대 생활 안내 자료표 (12개 시설 표준 데이터셋)
| 연번 | 시설명 | 위치 | 정규 운영 시간 | 예외 운영 조건 | 문의 연락처 | 주요 제공 업무 |
| :---: | :--- | :--- | :--- | :--- | :--- | :--- |
| 1 | 대대 의무실 | | | | | |
| 2 | 사지방 | | | | | |
| 3 | 체력단련실 | | | | | |
| ... | | | | | | |
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 02. 부대 안내 자료 정리하기", mission_02_prompt, mission_02_result)



---

## 🎯 미션 03. 질문에 답하는 프롬프트 작성하기
**소요시간**: 약 1.5시간

![미션 3 외부 지식 주입 RAG 기본 프롬프트 설계 다이어그램](/uploads/common/K09_m03_context_injection.png)

### 📋 미션 개요 및 배경
AI 모델 자체 지식에 의존하지 않고 제공된 부대 안내 자료에만 근거하여 답변하도록 제한하는 기본 지시 프롬프트를 설계합니다. 앞서 정리한 자료표를 컨텍스트로 제공하고 우선 질문 5개를 입력하여, 자료가 포함되었을 때와 없을 때의 답변 정확도 차이를 실증 비교합니다. 이를 통해 검색 증강(RAG)의 기본 개념과 컨텍스트 기반 질의응답 원리를 이해합니다.

### 🎯 미션 학습목표
'제공된 안내 자료에만 근거하여 답하라'는 제약 지시문과 자료표를 프롬프트에 포함하여 질문 5개를 질의하고, 자료가 없을 때의 응답과 정확도를 실증 비교합니다. **이를 통해 외부 지식 주입(Context Injection)으로 AI의 답변 범위를 제어하고 근거 기반 답변을 생성하게 하는 검색 증강(RAG) 프롬프트 설계 원리를 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 자료 근거 지시와 자료표를 넣은 프롬프트를 만든다.
#### 이 단계의 역할
AI가 외부 지식이나 환각에 의존하지 않고 제공된 자료표 내에서만 답변을 도출하도록 제약하는 컨텍스트 주입 프롬프트를 설계합니다.

#### 해야 할 일

<li>시스템 역할(Role), 답변 원칙(Constraints), 컨텍스트(미션2 자료표)를 모듈화하여 프롬프트 뼈대를 작성합니다.

</li><li>"반드시 아래 [부대 안내 자료]에 근거하여 답하고 자료에 없는 내용은 절대 추측하지 말라"는 앵커링 문구를 배치합니다.

</li><li>AI 도구 채팅창에 지시문과 자료표를 주입(Context Injection)할 준비를 마칩니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[자료 기반 답변 기본 프롬프트 설계]:** "[역할]: 너는 한빛대대 부대 생활 정보 안내 도우미야.
[지시사항]: 아래 제공된 [부대 안내 자료]의 내용만을 근거로 장병의 질문에 답해줘. 자료에 명시되지 않은 사실은 절대로 지어내거나 추측하지 마.

[부대 안내 자료]:
{미션2 자료표 전문}

[질문]: 의무실 진료 시간과 물리치료실 이용 방법을 알려줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **제약 조건 명시:** '자료에 있는 내용만 근거로 답할 것'이 프롬프트 상단에 명시되었는가?
- **자료표 무결성:** 12개 시설의 정보가 프롬프트 컨텍스트 내에 온전히 포함되었는가?

#### 핵심 확인 지표
<blockquote>자료 주입형 기본 안내 프롬프트 서식

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>자료 기반 답변 프롬프트 초안

</li>

#### 🔹 Step 2. 우선 질문 5개를 한다.
#### 이 단계의 역할
작성된 자료 기반 프롬프트에 미션1에서 선별한 우선 질문 5개를 순차적으로 질의하여 AI의 답변을 수집합니다.

#### 해야 할 일

<li>우선 질문 1번부터 5번까지 차례대로 프롬프트에 넣어 질의를 실행합니다.

</li><li>AI가 생성한 각 질문별 응답 텍스트 전문을 추출하여 기록표에 정리합니다.

</li><li>답변에서 인용한 시설명, 운영 시간, 위치 정보가 자료표에 있는 내용인지 예비 확인합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[우선 질문 5건 연속 질의]:** "[프롬프트 본문]: {Step 1 프롬프트}
[질문 1]: 의무실 진료 시간과 야간 진료 접수 방법 알려줘.
[질문 2]: 평일 일과 후 PX(충성마트) 이용 시간은 언제인가요?
[질문 3]: 사이버지식정보방 위치와 주말 이용 시간은?
[질문 4]: 체력단련실 샤워장 이용 가능한 시간 알려줘.
[질문 5]: 병영도서관 도서 대출 권수와 기간은 어떻게 되나요?"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **5개 질문 전수 질의:** 선정된 우선 질문 5건이 빠짐없이 질의되었는가?
- **자료 인용 여부:** AI가 자신의 상식이 아닌 제공된 자료표의 내용을 인용하여 답변했는가?

#### 핵심 확인 지표
<blockquote>우선 질문 5개에 대한 AI 1차 응답 기록표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>우선 질문 5개 AI 질의 응답 로그

</li>

#### 🔹 Step 3. 자료 없이 같은 질문을 해 비교한다.
#### 이 단계의 역할
부대 안내 자료 없이 동일한 5개 질문을 AI에 질의하고, 자료가 있을 때와 없을 때의 응답을 대조하여 컨텍스트 주입의 효과를 실증 비교합니다.

#### 해야 할 일

<li>새 대화창을 열고 안내 자료표 없이 "한빛대대 의무실 운영 시간을 알려줘" 등 동일 5개 질문을 질의합니다.

</li><li>자료가 없을 때 AI가 생성하는 막연한 일반론, 추측성 답변 및 허위 정보(환각)를 기록합니다.

</li><li>[자료 주입 답변 vs 자료 미제공 답변]의 구체성, 정확도, 환각 여부를 비교표로 정리합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[자료 미제공 비교 질의]:** "한빛대대 의무실 운영 시간과 위치, 야간 진료 접수 방법을 알려줘. (안내 자료표 주입 없이 단독 질문)"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **환각 현상 식별:** 자료가 없을 때 AI가 실제와 다른 가상의 시간이나 부서명을 지어냈는지 식별했는가?
- **비교표 완성도:** 5개 질문 각각에 대해 자료 유/무에 따른 답변 차이가 구체적으로 대조되었는가?

#### 핵심 확인 지표
<blockquote>자료 유무에 따른 AI 응답 비교표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>자료 유무 비교표(필수 산출물)

</li>

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할
자료표에서 특정 시설 정보를 일부 삭제한 부분 자료표를 주입하여, 부분적 정보 부재 시 AI의 반응을 스트레스 테스트합니다.

#### 해야 할 일
자료표 일부만 넣었을 때 어떤 답이 나오는지 확인해 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[부분 결손 자료표 반응 시험 질의]:** "[부분 자료표]: {체력단련실 정보가 삭제된 자료표}
[질문]: 체력단련실 헬스장 샤워실 운영 시간 알려줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **정보 경계 인지:** 삭제된 시설에 대해 AI가 지어내지 않고 정보가 없음을 올바르게 인지하는가?

#### 핵심 확인 지표
<blockquote>부분 결손 조건 스트레스 테스트 결과 메모

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>심화 도전 과제 수행 결과 메모

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M03] 질문에 답하는 프롬프트 작성하기 실습 가이드 및 작성 서식

## 1. 자료 기반 답변 프롬프트 서식 (Context Injection)

```markdown
[역할]: 너는 육군 가상 한빛대대 부대 생활 정보 질의응답 도우미야.
[지시사항]:
1. 반드시 아래 [부대 생활 안내 자료]에 명시된 사실에만 근거하여 장병의 질문에 답변하라.
2. 자료에 명시되지 않은 내용은 절대로 임의로 추측하거나 상상하여 지어내지 말라.
3. 친절하고 신뢰감 있는 군 경어체(합니다/습니다 체)로 답변하라.

[부대 생활 안내 자료]:
{미션2에서 정리한 12건 안내 자료표 전문}

[사용자 질문]:
{질문 입력}
```

---

## 2. 자료 유/무 비교 대조표 (5개 질문 실증 테스트)

| 질문 번호 | 질문 내용 | 자료 주입 시 AI 응답 (정확도) | 자료 미제공 시 AI 응답 (환각 발생 여부) | 분석 결과 |
| :---: | :--- | :--- | :--- | :--- |
| **TQ01** | 의무실 진료 시간 및 야간 접수 | 평일 09:00~17:30(소독 12~13:30 제외), 야간 당직 군의관 접수 (정확) | "통상 군 부대 의무실은 09시부터 17시까지..." 모호한 추측 및 가상 번호 날조 | 자료 주입 시 100% 팩트 일치 |
| **TQ02** | 평일 일과 후 PX 이용 시간 | 평일 12:00~19:30, 14~15시 재고조사 제외 (정확) | "일과 후 18:00~20:00까지 운영됩니다" 오답 생성 | 자료표의 예외 시간 정확 반영 |
| **TQ03** | 사지방 위치 및 주말 이용 시간 | 복지회관 2층 201호, 주말 09:00~21:00 (정확) | 위치를 본관 3층 등으로 허위 생성 | 건물 번호 및 호수까지 완벽 매칭 |
| **TQ04** | 체단실 이용 시간 및 방역 시간 | 평일 06:30~21:00, 수요일 14~16시 방역 (정확) | 방역 소독 시간 완전 누락 | 예외 조항 누락 방지 효과 확인 |
| **TQ05** | 도서관 대출 권수 및 기간 | 1인 3권, 14일간 대출 가능 (정확) | "보통 2권 7일간 대출 가능합니다" 일반론 왜곡 | 정확한 대출 규정 인용 확인 |

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **부대 생활 안내 자료 주입형 기본 프롬프트 템플릿 (MD):** 시스템 역할(Role), 자료 컨텍스트(Context), 제약조건(Constraints) 모듈형 지시문
- **우선 질문 5선 테스트 시트 및 질의응답 기록부:** 5개 핵심 질문별 질의 실행 및 AI 원시 응답 추출 양식
- **[학습 가이드] 외부 지식 주입(Context Injection)과 RAG 기초 원리:** 모델 자체 지식이 아닌 제공된 데이터셋에만 근거하여 답을 생성하게 제한하는 프롬프트 엔지니어링 원리

**실습 팁:** 프롬프트 상단에 "아래 [부대 안내 자료]에 없는 내용은 절대 추측하여 지어내지 말 것"이라는 강력한 앵커링 문구를 배치해야 환각을 예방할 수 있습니다.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] 프롬프트 작성(자료 근거) — 자료에만 근거하도록 제약 지시문을 설계하고 자료 유/무에 따른 답변 차이를 실증 비교했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>자료 주입형 기본 안내 프롬프트 서식 완성

</li><li>우선 질문 5건 전수 질의 실행 및 로그 수집

</li><li>자료 유/무에 따른 정확도 및 환각(Hallucination) 비교표 완성

</li>

#### 채점자 판정 메모
<blockquote>자료가 없을 때 AI가 임의로 지어낸 답변(예: 가상 부대 시간 왜곡)을 명확히 식별하고, 자료 주입의 효과를 입증했는지 확인한다.

</blockquote>





In [ ]:
# [미션 03. 질문에 답하는 프롬프트 작성하기] 학생 실습 프롬프트 및 결과 기록란
mission_03_prompt = """
# [TODO: 미션 03] Context Injection(자료 주입) 기법을 적용하여 12건의 자료표만을 근거로 답변하도록 시스템 프롬프트를 작성하세요.
# 힌트: 역할, 엄격한 근거 준수 지시, 12건 자료표, 그리고 5대 우선 질문을 순차적으로 포함합니다.
# (여기에 작성한 프롬프트를 입력하세요)
"""

mission_03_result = """
### 📋 [미션 03 실습 결과]
## 1. 자료 기반 질의응답 실증 테스트 (5개 질문)
| 질문 번호 | 질문 내용 | 자료 주입 시 AI 응답 | 자료 미제공 시 응답 (환각 여부) | 분석 결과 |
| :---: | :--- | :--- | :--- | :--- |
| **TQ01** | 의무실 진료 시간 및 야간 접수 | | | |
| **TQ02** | 사지방 주말 운영 시간 | | | |
| **TQ03** | 체단실 주말 샤워 온수 공급 | | | |
| **TQ04** | PX 평일 운영 시간 | | | |
| **TQ05** | 도서관 책 대출 권수 및 반납 | | | |
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 03. 질문에 답하는 프롬프트 작성하기", mission_03_prompt, mission_03_result)



---

## 🎯 미션 04. 답변이 자료와 맞는지 확인하기
**소요시간**: 약 1.5시간

![미션 4 원문 대조 검증 및 출처 표기 의무화 다이어그램](/uploads/common/K09_m04_factcheck_citation.png)

### 📋 미션 개요 및 배경
생성형 AI가 안내 자료를 잘못 해석하거나 누락·왜곡하여 발생하는 오안내 위험을 통제하기 위해 답변 대조 검증을 수행합니다. 생성된 답변의 운영 시간·장소·연락처를 원본 안내 자료와 1:1로 비교하여 '맞음/틀림/지어냄'으로 분류하고, 답변 끝에 반드시 근거 시설명을 표기하도록 프롬프트를 개선합니다. AI 결과를 맹신하지 않고 인간이 직접 검증하는 HITL(Human-in-the-Loop) 실무 습관을 확립합니다.

### 🎯 미션 학습목표
AI 답변의 시간·위치·연락처를 자료표와 1:1로 대조하여 맞음·틀림·지어냄으로 정량 판정하고, 답변 끝에 출처 근거(시설명)를 의무 표기하도록 지시문을 개선합니다. **이를 통해 AI의 환각(Hallucination) 현상을 식별·검증하는 팩트체크 기법과 출처 표기를 유도하는 프롬프트 통제 기술을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 답변을 대조표에 맞음·틀림·지어냄으로 표시한다.
#### 이 단계의 역할
미션3에서 수집된 AI 답변 문장을 단위 사실(시간, 장소, 대상, 조건)로 분해하여 원문 자료표와 1:1 대조 판정합니다.

#### 해야 할 일

<li>답변 대조표 서식을 준비하고 질문 5건의 AI 답변 문장을 단위 정보별로 분해합니다.

</li><li>원문 자료표와 대조하여 일치하는 정보는 '맞음', 수치 오차는 '틀림', 원문에 없는 창작은 '지어냄'으로 라벨링합니다.

</li><li>오류가 발생한 항목의 원인(프롬프트 지시 모호성, 수치 오독 등)을 정밀 분석합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[단위 사실 교차 검증 질의]:** "다음 [AI 생성 답변]을 아래 [부대 안내 자료]와 한 문장씩 비교 검증해 줘. 각 문장이 사실인지(맞음), 숫자가 다른지(틀림), 자료에 아예 없는 내용인지(지어냄) 판정하고 근거를 제시해 줘.
[부대 안내 자료]: {미션2 자료표}
[AI 생성 답변]: {미션3 생성 답변}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **엄밀한 육안 대조:** 시간(분 단위), 요일, 장소 명칭이 글자 그대로 일치하는지 육안으로 재확인했는가?
- **판정의 객관성:** 환각(지어냄) 요소가 엄격하게 감지되었는가?

#### 핵심 확인 지표
<blockquote>답변 문장별 맞음/틀림/지어냄 판정 대조표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>답변 대조표(필수 산출물)

</li>

#### 🔹 Step 2. 틀린 답의 질문 표현을 기록한다.
#### 이 단계의 역할
어떤 유형의 질문 표현(모호한 질문, 복합 질문, 군 축약어)이 AI의 오답이나 왜곡을 유발했는지 상관관계를 도출합니다.

#### 해야 할 일

<li>대조표에서 '틀림' 또는 '지어냄'으로 판정된 질문들의 표현 방식을 분석합니다.

</li><li>두 가지를 한 번에 물어본 경우(복합 질문)나 시설 명칭을 줄여 부른 경우 등 오류 유발 패턴을 기록합니다.

</li><li>장병들이 질문을 작성할 때 유의해야 할 사용자 가이드라인 초안을 정리합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[오류 유발 질문 패턴 분석 질의]:** "장병들이 입력한 다음 질문 표현들 중 AI가 잘못된 답변을 생성하게 만든 표현상 취약점(모호성, 복합 질의, 약어 등)을 분석하고, 이를 방지할 수 있는 명확한 질문 변환 예시를 작성해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **오류 원인 규명:** 단순 오답 표기가 아닌 질문 구조적 원인이 규명되었는가?
- **개선 방향 도출:** 질문 표현을 어떻게 고쳐야 정답률이 올라가는지 해결책이 제시되었는가?

#### 핵심 확인 지표
<blockquote>질문 표현별 오류 발생 원인 분석표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>틀린 답의 질문 표현 분석 기록지

</li>

#### 🔹 Step 3. 근거 시설명을 쓰게 한다.
#### 이 단계의 역할
답변의 신뢰도를 높이고 검증을 용이하게 하기 위해 답변 끝에 반드시 인용한 '근거 항목(시설명)'을 명기하도록 지시문을 개선합니다.

#### 해야 할 일

<li>프롬프트 지시사항에 "답변 말미에 [근거: OOO 안내] 형식으로 출처 시설명을 표기하라"는 규칙을 추가합니다.

</li><li>개선된 프롬프트로 5개 질문을 재실행하여 출처 표기가 올바르게 출력되는지 확인합니다.

</li><li>표기된 근거 시설명이 실제 원문의 해당 섹션과 정확히 일치하는지 대조합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[출처 표기 의무화 프롬프트 개선]:** "[개선 지시사항 추가]: ...반드시 답변의 마지막 줄에 [근거 시설: 시설명] 형태로 정보를 인용한 원본 시설 명칭을 명시해야 함. 출처를 특정할 수 없는 내용은 답변에 포함하지 말 것.
[질문]: PX 이용 시간 알려줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **출처 표기 준수율:** 모든 답변 끝에 지정된 형식([근거 시설: ...])이 누락 없이 출력되었는가?
- **출처 일치성:** 답변 내용과 표기된 근거 시설명이 실제로 부합하는가?

#### 핵심 확인 지표
<blockquote>근거 시설명 표기 의무화 프롬프트

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>근거 표기 프롬프트(필수 산출물)

</li>

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할
근거 시설명뿐만 아니라 해당 자료의 행 번호(Row ID)까지 더욱 정밀하게 인용하도록 프롬프트를 고도화합니다.

#### 해야 할 일
답변 끝에 근거뿐만 아니라 해당 시설의 대표 업무 1가지를 덧붙이게 해 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[행 번호 정밀 인용 질의]:** "답변 작성 시 반드시 [근거: No.03 의무실 진료 안내]처럼 자료표의 행 번호와 시설명을 함께 표기하도록 프롬프트를 수정해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **정밀 매핑:** 자료표 행 번호와 시설명이 오차 없이 정확히 매핑되었는가?

#### 핵심 확인 지표
<blockquote>정밀 출처 표기 프롬프트 및 검증 시간 단축 메모

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>심화 최적화 메모

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M04] 답변이 자료와 맞는지 확인하기 실습 가이드 및 작성 서식

## 1. AI 답변-원문 1:1 교차 대조표 (Fact-checking Sheet)

| 문항 | AI 답변 문장 분해 | 원본 자료표 대조 내용 | 판정 (맞음/틀림/지어냄) | 오류 원인 및 보완 사항 |
| :--- | :--- | :--- | :---: | :--- |
| **Q1-1** | "의무실은 본관 1층 서편에 위치합니다." | 본관 1층 서편 일치 | **맞음** | 사실 일치 |
| **Q1-2** | "평일 진료는 09:00~17:30까지입니다." | 평일 09:00~17:30 일치 | **맞음** | 수치 일치 |
| **Q1-3** | "12:00~13:30은 소독 시간입니다." | 12:00~13:30 소독 일치 | **맞음** | 조건 일치 |
| **Q2-1** | "충성마트는 평일 12:00~20:00까지입니다." | 원문은 19:30 마감 | **틀림** | AI가 20:00로 숫자 왜곡 |
| **Q2-2** | "주말에는 언제든 자유롭게 방문 가능합니다."| 원문: 일요일 휴무 | **지어냄** | 일요일 휴무 조건 누락 및 창작 |

---

## 2. 근거(출처) 표기 의무화 프롬프트 템플릿

```markdown
[지시사항 수정/추가]:
- 반드시 답변의 마지막 줄에 [근거 시설: OO 안내] 형식으로 참조한 자료표의 정확한 시설 명칭을 명기할 것.
- 인용한 출처를 특정할 수 없는 내용은 답변에 일체 포함하지 말 것.

[출력 예시]:
답변: 대대 의무실 일반 진료는 평일 09:00부터 17:30까지입니다. (12:00~13:30 소독 제외)
[근거 시설: 대대 의무실 운영 안내]
```

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **단위 사실 교차 검증 대조표 양식 (Fact-checking Sheet):** 시간, 위치, 연락처 정보별 '맞음/틀림/지어냄' 1:1 판정 서식
- **출처(근거) 표기 의무화 프롬프트 템플릿:** 답변 말미에 [근거: 시설명]을 강제 출력하도록 설계된 개선 프롬프트
- **[학습 가이드] AI 환각(Hallucination) 감지 및 원문 대조 검증(HITL) 수칙:** AI의 왜곡·날조 패턴 유형별 식별 요령 및 장병 직접 검증 습관 가이드

**실습 팁:** AI 답변에서 분 단위 숫자(예: 17:30 vs 18:00)나 요일 조건은 대충 넘어가기 쉬우므로, 원문 글자를 하나씩 손가락으로 짚어가며 육안 검증하세요.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] AI 결과 검증 — AI 답변 문장을 원문과 1:1 대조하여 오류를 식별하고 출처 근거를 표기하도록 프롬프트를 개선했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>답변 문장별 맞음/틀림/지어냄 1:1 대조 점검표 완성

</li><li>오류 유발 질문 표현 패턴 분석 완료

</li><li>답변 끝 근거 시설명 표기 프롬프트 개선 완료

</li>

#### 채점자 판정 메모
<blockquote>단순히 맞다고 넘어가지 않고, 분 단위 시간이나 요일 조건의 미세한 왜곡을 짚어내었는지 대조표의 판정 근거를 확인한다.

</blockquote>





In [ ]:
# [미션 04. 답변이 자료와 맞는지 확인하기] 학생 실습 프롬프트 및 결과 기록란
mission_04_prompt = """
# [TODO: 미션 04] AI 답변의 문장별 단위 사실을 원문과 대조 검증(Fact-checking)하는 프롬프트를 작성하세요.
# 힌트: 맞음/틀림/지어냄 판정 기준과 인용 근거 표기 규칙을 포함합니다.
# (여기에 작성한 프롬프트를 입력하세요)
"""

mission_04_result = """
### 📋 [미션 04 실습 결과]
## 1. AI 답변-원문 1:1 교차 대조표 (Fact-checking)
| 문항 | AI 답변 문장 분해 | 원본 자료표 대조 내용 | 판정 (맞음/틀림/지어냄) | 오류 원인 및 보완 사항 |
| :--- | :--- | :--- | :---: | :--- |
| **Q1-1** | | | | |
| **Q1-2** | | | | |
| **Q1-3** | | | | |
| **Q1-4** | | | | |

## 2. 근거(출처) 표기 지시문 보완 결과
- 보완된 프롬프트 지침:
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 04. 답변이 자료와 맞는지 확인하기", mission_04_prompt, mission_04_result)



---

## 🎯 미션 05. 자료에 없는 질문 처리하기
**소요시간**: 약 1.5시간

![미션 5 자료 밖 질문 차단 및 안전 Fallback 규칙 다이어그램](/uploads/common/K09_m05_safety_fallback.png)

### 📋 미션 개요 및 배경
안내 자료에 명시되지 않은 질문이 입력되었을 때 AI가 허위 답변을 생성(할루시네이션)하는 현상을 방지하는 예외 처리 규칙을 수립합니다. '월급일', '휴무 행사' 등 자료 밖 질문을 입력하여 AI의 오작동 패턴을 확인하고, 정보 부재 시 "자료에 없음" 명시와 함께 행정반·당직실 등 공식 부서로 연계하도록 지시문을 고도화합니다. 시스템의 안전성과 신뢰성을 확보하는 거절 프롬프팅 기법을 습득합니다.

### 🎯 미션 학습목표
'월급날', '행사 일정' 등 자료 밖 질문을 입력하여 AI의 오작동 패턴을 관찰하고, 정보 부재 시 임의 추론을 차단하며 "자료에 없음" 명시 및 담당 부서를 안내하도록 규칙을 추가합니다. **이를 통해 시스템의 안전 경계를 설정하고 오안내 위험을 선제 차단하는 안전 프롬프팅 및 Fallback(예외 대응) 처리 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 질문 카드 17~19를 넣는다.
#### 이 단계의 역할
안내 자료에 전혀 수록되지 않은 질문 카드 17~19번(월급일, 식단표, 체육대회 일정 등)을 입력하여 AI의 허위 답변 생성을 포착합니다.

#### 해야 할 일

<li>실습 데이터에서 '자료 밖 질문'으로 설계된 카드 17, 18, 19번을 확인합니다.

</li><li>현재 프롬프트에 17~19번 질문을 입력하고 AI의 응답을 수집합니다.

</li><li>AI가 자료에 없음에도 당당하게 가상의 날짜나 일정을 창작해 내는 환각 응답을 기록합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[자료 밖 질문 취약점 노출 질의]:** "[프롬프트 본문]: {기존 자료 기반 프롬프트}
[자료 밖 질문 17]: 이번 달 병사 급여 입금일과 시간을 알려줘.
[자료 밖 질문 18]: 다음 주 연대 체육대회 축구 일정과 대진표를 알려줘.
[자료 밖 질문 19]: 부대 내 종교시설 세례식 일정이 어떻게 되나요?"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **환각 현상 포착:** 자료에 없는 내용임에도 AI가 임의로 그럴듯한 답변을 지어냈는지 확인했는가?
- **위험성 인지:** 이러한 거짓 답변이 부대 행정 혼선에 미치는 위험성을 기록했는가?

#### 핵심 확인 지표
<blockquote>자료 밖 질문 3건에 대한 AI 환각 응답 기록표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>자료 밖 질문 1차 질의 결과 기록지

</li>

#### 🔹 Step 2. 자료에 없으면 안내 문구로 답하게 조건을 넣는다.
#### 이 단계의 역할
정보 부재 시 임의 추론 생성을 즉각 중단하고 표준 거절 문구와 함께 실무 담당 부서(행정반/당직실)로 연결하는 Fallback 규칙을 설계합니다.

#### 해야 할 일

<li>"자료에 해당 정보가 전혀 없을 경우 '해당 정보는 안내 자료에 없습니다'라고 출력하라"는 원칙을 명문화합니다.

</li><li>"추가 확인이 필요할 경우 [담당 문의처: 본부 행정반 033-XXX-XXXX 또는 당직실]로 문의하라"는 고정 안내문을 작성합니다.

</li><li>프롬프트의 [예외 대응 규칙] 섹션에 해당 Fallback 지침을 정식 삽입합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[예외 대응 Fallback 프롬프트 설계]:** "[예외 처리 규칙 추가]:
- 사용자의 질문이 위 [부대 안내 자료]에 언급되지 않은 내용이라면 절대 임의로 지어내지 마십시오.
- 자료에 없는 질문에는 반드시 다음과 같이 고정된 서식으로 답하십시오:
"⚠️ 안내 자료 미등록 정보: 해당 문의는 부대 생활 안내 자료에 등록되어 있지 않습니다. 정확한 규정 및 일정은 본부 행정반 또는 지휘통제실(당직실)로 문의하시기 바랍니다.""

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **거절 원칙 명확성:** 애매한 추측성 답변 없이 단호하게 정보 부재를 인정하도록 지시되었는가?
- **공식 창구 안내:** 실무 장병이 찾아갈 수 있는 공식 담당 부서(행정반/당직실)가 명시되었는가?

#### 핵심 확인 지표
<blockquote>자료 밖 질문 처리 규칙(Fallback 지침문)

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>자료 밖 질문 처리 규칙(필수 산출물)

</li>

#### 🔹 Step 3. 전후를 비교한다.
#### 이 단계의 역할
예외 처리 규칙 적용 전(환각 답변)과 적용 후(안전 거절 답변)를 1:1로 비교하여 오안내 위험 차단 성과를 입증합니다.

#### 해야 할 일

<li>규칙 적용 후 동일한 17~19번 질문을 재질의하여 새 응답을 받습니다.

</li><li>적용 전의 허위 답변과 적용 후의 표준 거절 답변을 나란히 배치한 전후 비교표를 작성합니다.

</li><li>정상 질문(1~5번)을 다시 넣어 예외 규칙 추가로 인해 기존 정상 답변 기능이 훼손되지 않았는지(회귀 테스트) 확인합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[예외 처리 적용 전후 비교 질의]:** "다음 질문 3건에 대해 예외 처리 규칙이 적용된 프롬프트로 재응답하고, 이전 응답과의 안전성 개선 사항을 비교 요약해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **환각 차단율 100%:** 3개 질문 모두에서 지어낸 정보가 완벽히 소멸되었는가?
- **정상 기능 유지:** 기존에 잘 답변하던 정상 질문 5건이 여전히 온전하게 출력되는가?

#### 핵심 확인 지표
<blockquote>자료 밖 질문 처리 전후 비교 검증표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>처리 전후 비교(필수 산출물)

</li>

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할
자료에 있는 정보와 없는 정보가 섞인 복합 질문(혼합 질문)에 대해 분기 처리하는 고급 프롬프팅을 테스트합니다.

#### 해야 할 일
'자료에 없으면 가장 가까운 시설을 안내하라'는 조건을 추가해 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[혼합 질문 분기 처리 질의]:** "[혼합 질문 처리 규칙]: 한 질문 안에 자료에 있는 내용과 없는 내용이 섞여 있을 경우, 자료에 있는 부분은 정상 답변하고 없는 부분만 골라 'OO 정보는 자료에 없어 행정반 문의가 필요합니다'로 분리 서술할 것.
[질문]: 의무실 진료 시간은 몇 시까지이고, 다음 주 사격 훈련 일정은 언제인가요?"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **부분 분기 정확성:** 알고 있는 정보와 모르는 정보가 뒤섞이지 않고 깔끔히 분리되었는가?

#### 핵심 확인 지표
<blockquote>복합 혼합 질문 분기 처리 시험 메모

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>심화 도전 과제 수행 메모

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M05] 자료에 없는 질문 처리하기 실습 가이드 및 작성 서식

## 1. 자료 밖 질문 처리 규칙 (Fallback 템플릿)

```markdown
[예외 대응 처리 수칙]:
1. 사용자의 질문 내용이 제공된 [부대 생활 안내 자료]에 명시되어 있지 않은 경우, 절대 임의로 상상하거나 추론하여 지어내지 말라.
2. 자료에 없는 질문에는 반드시 아래의 표준 서식으로 단호하게 답변하라:
   "⚠️ 안내 자료 미등록 정보: 해당 문의 사항은 부대 생활 안내 자료에 등록되어 있지 않습니다. 정확한 규정 및 세부 일정은 본부 행정반(내선 101) 또는 당직사령실로 문의하시기 바랍니다."
```

---

## 2. 예외 처리 규칙 적용 전후 비교 검증표

| 질문 ID | 질문 내용 | 적용 전 AI 답변 (환각 현상) | 적용 후 AI 답변 (안전 Fallback) | 판정 |
| :---: | :--- | :--- | :--- | :---: |
| **Q17** | 병사 월급 입금일 | "매월 10일 정오에 나라사랑카드 계좌로 입금됩니다." (가상 날조) | "⚠️ 안내 자료 미등록 정보: ...본부 행정반 문의" | **PASS** |
| **Q18** | 연대 체육대회 일정 | "다음 달 15일 연병장에서 축구 예선이 열립니다." (가상 날조) | "⚠️ 안내 자료 미등록 정보: ...본부 행정반 문의" | **PASS** |
| **Q19** | 개방행사 면회실 예약 | "최소 1주일 전까지 신청서를 내야 합니다." (임의 추론) | "⚠️ 안내 자료 미등록 정보: ...본부 행정반 문의" | **PASS** |

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **자료 범위 밖 예외 질문 카드 3종 (카드 17~19):** 월급일, 체육대회 일정, 세례식 등 자료표에 없는 함정 질의 세트
- **안전 Fallback(예외 대응) 처리 규칙 템플릿:** 정보 부재 시 "자료에 없음" 명시 및 행정반/당직실 공식 안내문 서식
- **[학습 가이드] 안전 프롬프팅 및 Fallback(거절) 메커니즘:** AI가 모르는 것을 솔직하게 인정하고 공식 채널로 토스하게 만드는 프롬프트 안전 경계 설계법

**실습 팁:** 거절 지침을 줄 때는 단순히 "답하지 마"라고만 하면 AI가 침묵하거나 일반론을 말하므로, 반드시 "정확히 이 문장으로 답하라"며 표준 거절 문구를 지정해 주어야 합니다.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] 예외 처리·환각 방지 — 자료 밖 질문에 대해 거짓 정보를 지어내지 않고 표준 문구 및 담당 부서를 안내하는 Fallback 규칙을 구현했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>자료 밖 질문 3건에 대한 취약점 포착 및 응답 기록

</li><li>공식 문의처(행정반/당직실)가 연계된 표준 Fallback 지침 수립

</li><li>예외 규칙 적용 전후 비교표 작성을 통한 환각 차단율 100% 달성

</li>

#### 채점자 판정 메모
<blockquote>자료 밖 질문(월급일, 체육대회 등)을 던졌을 때 단 한 문장의 거짓 정보도 생성하지 않고 완벽히 거절 및 공식 부서로 토스하는지 확인한다.

</blockquote>





In [ ]:
# [미션 05. 자료에 없는 질문 처리하기] 학생 실습 프롬프트 및 결과 기록란
mission_05_prompt = """
# [TODO: 미션 05] 자료 범위 밖 예외 질문(Out-of-scope)에 대한 Fallback 방어 지시문을 작성하세요.
# 힌트: 자료에 없는 내용은 상상하지 않고 정해진 안내 문구를 출력하도록 명시합니다.
# (여기에 작성한 프롬프트를 입력하세요)
"""

mission_05_result = """
### 📋 [미션 05 실습 결과]
## 1. 자료 밖 질문 처리 규칙 적용 전후 비교
| 질문 번호 | 자료 밖 질문 내용 | 규칙 적용 전 (환각 발생 내용) | 규칙 적용 후 (안전 대응 결과) | 판정 |
| :---: | :--- | :--- | :--- | :---: |
| **Q17** | 병사 급여 입금일 및 시간 | | | |
| **Q18** | 연대 체육대회 축구 일정 | | | |
| **Q19** | 체단실 바벨 무게 추가 계획 | | | |
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 05. 자료에 없는 질문 처리하기", mission_05_prompt, mission_05_result)



---

## 🎯 미션 06. 답변 형식 통일하기
**소요시간**: 약 1.5시간

![미션 6 Few-shot 모범 예시 기반 5단 표준 출력 서식 제어 다이어그램](/uploads/common/K09_m06_format_fewshot.png)

### 📋 미션 개요 및 배경
질문자의 다양한 말투나 축약어에도 흔들리지 않고 규격화된 답변을 제공하도록 출력 포맷을 일원화합니다. '답변/위치/운영시간/문의처/근거' 템플릿과 Few-shot(예시)을 프롬프트에 포함하고, 반말·줄임말·오타 질문에서도 지정된 형식이 준수되는지 스트레스 테스트를 진행합니다. 사용자 경험(UX)의 일관성을 보장하고 정보 가독성을 극대화하는 출력 제어 기법을 익힙니다.

### 🎯 미션 학습목표
'답 / 위치 / 시간 / 문의 / 근거' 표준 출력 형식을 정의하고 예시 1개(Few-shot)를 프롬프트에 반영한 뒤, 반말·줄임말·오타 질문에서도 지정된 형식이 유지되는지 점검합니다. **이를 통해 사용자 입력의 다양성에도 일관된 규격으로 응답을 강제하는 출력 포맷 제어 기법과 프롬프트 견고성(Robustness) 검증 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 답변 형식(답·위치·시간·문의·근거)을 정한다.
#### 이 단계의 역할
모든 답변이 일관된 시각적 구조를 유지하도록 [답변 요약 / 위치 / 운영 시간 / 문의처 / 근거] 5단 표준 출력 서식을 설계합니다.

#### 해야 할 일

<li>장병들이 모바일이나 PC 화면에서 한눈에 정보를 파악할 수 있는 5개 필수 항목 서식을 정의합니다.

</li><li>각 항목별 줄바꿈, 구분 기호, 글머리 기호 규칙을 표준화합니다.

</li><li>프롬프트의 [출력 서식 지정] 섹션에 해당 템플릿을 명시합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[5단 표준 답변 서식 템플릿 설계]:** "[출력 포맷 규칙]: 모든 질문에 대해 반드시 아래 서식을 엄격히 준수하여 응답할 것:
📌 요약 답변: [한 문장으로 핵심 답변]
📍 시설 위치: [건물명 및 층수]
⏰ 운영 시간: [정규 시간 및 예외 시간]
📞 문의 연락: [내선 번호 또는 담당자]
🔍 인용 근거: [참조한 자료표의 시설명]"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **5대 항목 완전성:** 답변, 위치, 시간, 문의, 근거 5대 항목이 누락 없이 정의되었는가?
- **시각적 일관성:** 글머리 기호와 줄바꿈이 표준화되어 있는가?

#### 핵심 확인 지표
<blockquote>5단 표준 답변 출력 서식 명세서

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>답변 형식(필수 산출물)

</li>

#### 🔹 Step 2. 예시 1개를 넣는다.
#### 이 단계의 역할
AI가 출력 서식을 정확히 모방하도록 이상적인 질의응답 샘플 1쌍을 프롬프트에 주입하는 Few-shot 프롬프팅을 적용합니다.

#### 해야 할 일

<li>'의무실 진료'에 대한 모범 질의와 5단 서식이 완벽히 적용된 모범 응답 예시(Shot) 1건을 작성합니다.

</li><li>프롬프트 본문의 지시사항 바로 아래에 [질의응답 예시] 블록으로 삽입합니다.

</li><li>다른 시설 질문(사이버지식정보방 등)을 입력하여 예시의 형식을 완벽하게 계승하는지 확인합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[Few-shot 예시 주입 프롬프트]:** "[질의응답 Few-shot 예시]:
질문: 의무실 진료 언제까지 하나요?
답변:
📌 요약 답변: 평일 17:30까지 일반 진료가 가능합니다.
📍 시설 위치: 본관 1층 의무실
⏰ 운영 시간: 평일 09:00~17:30 (점심 소독 12:00~13:30 제외, 야간은 당직 접수)
📞 문의 연락: 내선 103 (의무대)
🔍 인용 근거: [의무실 운영 안내]"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **예시 모방도:** 예시로 제시한 글머리 기호, 줄바꿈, 어투(합니다 체)를 AI가 정확히 모방하는가?
- **형식 붕괴 방지:** 새로운 질문에 대해서도 줄바꿈이 흐트러지지 않는가?

#### 핵심 확인 지표
<blockquote>1건의 Few-shot 예시가 포함된 서식 고정 프롬프트

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>예시를 넣은 프롬프트(필수 산출물)

</li>

#### 🔹 Step 3. 반말·줄임말·오타 질문으로 시험한다.
#### 이 단계의 역할
사용자가 반말("사지방 몇시까지임?"), 군 축약어("체단실 어디?"), 오타("의무실 위치좀요오")로 거칠게 질문해도 정중하고 규격화된 서식을 유지하는지 스트레스 테스트를 수행합니다.

#### 해야 할 일

<li>정상 질문 3건을 반말, 줄임말, 자음 탈락/오타 질문으로 변형한 테스트 세트를 만듭니다.

</li><li>변형된 질문들을 순차적으로 프롬프트에 입력하고 출력 결과를 수집합니다.

</li><li>질문의 어투에 휘말리지 않고 항상 군 표준 존댓말과 5단 고정 서식을 준수하는지 판정표를 작성합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[비정형 입력 스트레스 테스트 질의]:** "[변형 테스트 질문 1]: 야 사지방 몇시까지 하냐?
[변형 테스트 질문 2]: 체단실 위치 어딧는지 알려줭 ㅠㅠ
[변형 테스트 질문 3]: 츙성마트 주말에도 염?"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **어조의 일관성:** 사용자 질문이 반말이라도 AI는 군 표준 존댓말로 예의 바르게 응답하는가?
- **포맷 붕괴 방지:** 오타나 은어 질문에도 5단 서식이 깨지지 않고 유지되는가?

#### 핵심 확인 지표
<blockquote>변형 질문 3종 스트레스 테스트 결과표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>표현을 바꾼 질문 시험 결과(필수 산출물)

</li>

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할
모바일 화면에서 한눈에 볼 수 있도록 핵심 정보만 2줄로 축약하는 모바일 전용 초간결 서식 옵션을 설계합니다.

#### 해야 할 일
'한 줄 요약' 칸을 맨 위에 추가해 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[모바일 초간결 요약 서식 질의]:** "[모바일 간결 서식 규칙]: 스마트폰 화면용 초간결 모드 요청 시, [시설명 | 위치 | 운영시간]을 한 줄 표 형태로 압축 출력할 것."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **압축률 및 가독성:** 작은 모바일 화면에서도 스크롤 없이 핵심 정보가 읽히는가?

#### 핵심 확인 지표
<blockquote>모바일 초간결 서식 변환 메모

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>심화 서식 최적화 메모

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M06] 답변 형식 통일하기 실습 가이드 및 작성 서식

## 1. 5단 표준 답변 포맷 명세서
- 장병들이 모바일과 PC 화면에서 정보를 직관적으로 스캔할 수 있도록 일원화된 5개 고정 항목입니다.

```markdown
📌 요약 답변: [한 문장으로 결론 제시]
📍 시설 위치: [건물명 및 층수/호수]
⏰ 운영 시간: [정규 운영 시간 및 예외 조건]
📞 문의 연락: [담당 부서 및 내선 번호]
🔍 인용 근거: [참조한 자료표의 시설명]
```

---

## 2. Few-shot 모범 예시 탑재 프롬프트

```markdown
[지시사항]: 아래 질의응답 예시의 서식과 어조를 100% 동일하게 모방하여 응답하라.

[모범 예시]:
질문: 의무실 진료 언제까지 하나요?
답변:
📌 요약 답변: 평일 17:30까지 일반 진료가 가능합니다.
📍 시설 위치: 본관 1층 서편 의무실
⏰ 운영 시간: 평일 09:00~17:30 (점심 소독 12:00~13:30 제외, 야간은 당직 접수)
📞 문의 연락: 내선 103 (의무대)
🔍 인용 근거: [대대 의무실 운영 안내]
```

---

## 3. 비정형 입력(반말·줄임말·오타) 스트레스 테스트 결과표

| 테스트 유형 | 입력 질문 | AI 출력 결과 (서식 준수 및 어조 검증) | 판정 |
| :--- | :--- | :--- | :---: |
| **반말 질문** | "야 사지방 몇시까지 하냐?" | 📌 요약 답변: 사이버지식정보방은 평일 21:00까지 이용하실 수 있습니다.<br>📍 시설 위치: 복지회관 2층 201호... (군 표준 경어체 및 5단 서식 준수) | **PASS** |
| **줄임말 질문** | "체단실 위치랑 샤워장 정보 좀" | 📌 요약 답변: 체력단련실은 본관 지하 1층에 위치하며 샤워장을 갖추고 있습니다.<br>⏰ 운영 시간: 평일 06:30~21:00... | **PASS** |
| **오타 질문** | "츙성마트 주말에도 문 염?" | 📌 요약 답변: 충성마트(PX)는 토요일 10:00~18:00까지 운영하며 일요일은 휴무입니다.<br>📍 시설 위치: 복지회관 1층... | **PASS** |

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **5단 표준 답변 포맷 명세서:** [📌 요약 답변 / 📍 위치 / ⏰ 운영 시간 / 📞 문의 / 🔍 근거] 템플릿
- **Few-shot 모범 질의응답 예시문 (의무실 안내 샘플):** AI의 어조와 글머리 기호, 줄바꿈을 100% 통제하는 모범 샷(Shot)
- **[학습 가이드] Few-shot 프롬프팅과 비정형 입력 스트레스 테스트:** 사용자 질문이 반말, 줄임말, 오타여도 규격화된 정중한 답변을 강제하는 프롬프트 견고성(Robustness) 검증법

**실습 팁:** Few-shot 예시에 들어가는 이모지(📌, 📍, ⏰ 등)와 들여쓰기 형태를 그대로 유지하라고 지시문에 한 줄 명시하면 포맷 통일성이 비약적으로 상승합니다.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] 출력 형식 지정 — Few-shot 예시를 활용하여 5단 표준 서식을 구축하고 비정형(반말·줄임말·오타) 질문에서도 일관성을 유지했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>5단 표준 답변 포맷 명세서 정의 완료

</li><li>모범 질의응답 예시가 포함된 Few-shot 프롬프트 완성

</li><li>반말, 줄임말, 오타 질문 3종 스트레스 테스트 전수 통과

</li>

#### 채점자 판정 메모
<blockquote>사용자가 거칠게 반말로 질문해도 AI가 품격 있는 군 경어체와 5단 규격 서식을 칼같이 지켜내는지 확인한다.

</blockquote>





In [ ]:
# [미션 06. 답변 형식 통일하기] 학생 실습 프롬프트 및 결과 기록란
mission_06_prompt = """
# [TODO: 미션 06] 5단 표준 포맷(요약, 위치, 시간, 연락, 근거)을 적용한 Few-shot 프롬프트를 작성하세요.
# 힌트: 사용자가 한눈에 정보를 파악할 수 있도록 고정된 5대 항목 템플릿과 모범 예시를 제공합니다.
# (여기에 작성한 프롬프트를 입력하세요)
"""

mission_06_result = """
### 📋 [미션 06 실습 결과]
## 1. 5단 표준 포맷 적용 질의응답 결과
- **질문**: 주말에 사지방에서 인터넷 강의 들으려는데 몇 시까지 하고 어디로 가면 되나요?
- **AI 출력 결과**:
  - 📌 요약 답변:
  - 📍 시설 위치:
  - ⏰ 운영 시간:
  - 📞 문의 연락:
  - 🔍 인용 근거:
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 06. 답변 형식 통일하기", mission_06_prompt, mission_06_result)



---

## 🎯 미션 07. 신규 전입 병사용 안내 만들기
**소요시간**: 약 1.5시간

![미션 7 신병 적응 타임라인 구조화 및 요약 안내문 제작 다이어그램](/uploads/common/K09_m07_timeline_guide.png)

### 📋 미션 개요 및 배경
구축된 부대 생활 데이터와 질의응답 도우미를 활용하여 신규 전입 장병이 첫 주에 필수 숙지해야 할 행동 요령 안내문을 작성합니다. 전입 당일부터 첫 주말까지의 일과·시설 이용 정보를 시간 순서대로 재배열하고, 군 특유의 낯선 행정 용어를 알기 쉬운 표준어로 윤문합니다. 생성형 AI를 활용한 실용적 부대 적응 교육 콘텐츠 제작 역량을 완성합니다.

### 🎯 미션 학습목표
구축된 도우미와 안내 자료를 활용하여 전입 첫날부터 첫 주까지 알아야 할 생활 정보를 시간 순으로 재배치하고, 난해한 군 행정 용어를 쉬운 표현으로 다듬은 한 장짜리 요약 안내문을 제작합니다. **이를 통해 최종 사용자의 눈높이에 맞춰 정보를 재구성하고 AI를 활용해 실용적인 맞춤형 행정 콘텐츠를 기획·제작하는 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 첫날부터 첫 주까지 필요한 정보를 순서대로 정리하게 한다.
#### 이 단계의 역할
정리된 부대 생활 데이터를 신병의 시점(전입 1일 차 ~ 7일 차 주말)에 맞춘 시간 순(Timeline) 라이프사이클로 재정렬합니다.

#### 해야 할 일

<li>신병의 자대 도착 당일(입소 및 관물대 정리), 2~3일 차(시설 위치 파악 및 행정 등록), 첫 주말(종교 행사 및 여가 복지) 단계로 시간 축을 설정합니다.

</li><li>구축된 12개 시설 및 행정 절차 정보를 시간대별 필요도에 따라 매핑합니다.

</li><li>AI에게 신병 동선에 최적화된 시간순 타임라인 테이블 초안 생성을 요청합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[신병 적응 타임라인 구조화 질의]:** "너는 신병 교육대 훈육관이자 자대 적응 멘토야. 아래 [한빛대대 생활 안내 자료]를 바탕으로, 신병이 자대에 전입 온 첫날(Day 1), 2~3일 차, 첫 주말(Day 6~7)까지 겪게 되는 시간 순서대로 꼭 알아야 할 생활 수칙과 시설 이용 정보를 단계별로 재배열해 줘.
[안내 자료]: {미션2 자료표}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **시간순 타당성:** 첫날 당장 불필요한 주말 정보 등이 후순위로 적절히 배치되었는가?
- **핵심 동선 망라:** 첫날 생활관, 식당, 의무실 등 필수 시설이 누락 없이 포함되었는가?

#### 핵심 확인 지표
<blockquote>신병 첫 주 적응 타임라인 구조도

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>시간순 생활 정보 정리표

</li>

#### 🔹 Step 2. 한 장짜리 안내문으로 만든다.
#### 이 단계의 역할
타임라인으로 정리된 방대한 정보를 신병 관물대나 수첩에 부착하여 한눈에 볼 수 있는 1-Page(한 장) 요약 브로슈어 형태로 편집합니다.

#### 해야 할 일

<li>핵심 정보만을 엄선하여 1페이지 내에 들어갈 수 있는 마크다운 레이아웃(헤더, 요약 카드, 표, 체크리스트)을 설계합니다.

</li><li>AI에게 불필요한 미사여구를 제거하고 정보 밀도가 높은 한 장짜리 안내문 작성을 지시합니다.

</li><li>분량이 A4 1장을 초과하지 않도록 압축률을 검토합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[1-Page 신병 안내문 작성 질의]:** "앞서 정리한 신병 첫 주 생활 정보를 신병 수첩에 꽂고 다닐 수 있는 '한빛대대 신병 적응 가이드 (A4 1장 요약본)' 형태로 마크다운 문서를 작성해 줘. [1. 전입 당일 필수 체크, 2. 주요 시설 3대 핫스팟 운영 시간, 3. 긴급 시 도움 요청 창구] 3개 섹션으로 압축해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **분량 준수:** 군더더기 없이 A4 1장 분량(약 800~1,000자 내외)으로 압축되었는가?
- **시각적 레이아웃:** 섹션 구분이 명확하고 글머리 기호가 일목요연한가?

#### 핵심 확인 지표
<blockquote>1-Page 신규 전입 병사 첫 주 안내문 초안

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>한 장짜리 안내문 작성본 초안

</li>

#### 🔹 Step 3. 어려운 용어를 쉬운 말로 바꾼다.
#### 이 단계의 역할
사회 초년생인 신병이 이해하기 어려운 군대 은어, 한자어, 축약된 행정 용어를 알기 쉬운 표준어로 윤문합니다.

#### 해야 할 일

<li>안내문 초안에서 낯선 군사용어(사지방, 체단실, 총기수치, 조식/석식, 관물대 등)를 추출합니다.

</li><li>쉬운 대체 표현이나 괄호 속 용어 설명(예: 사이버지식정보방(인터넷 PC 카페))을 작성합니다.

</li><li>AI에게 신병 눈높이에 맞춘 친절하고 명확한 어조로 최종 교정을 요청합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[군 행정 용어 순화 및 윤문 질의]:** "다음 신병 안내문 초안에 포함된 낯선 군 행정 용어 및 축약어들을 찾아내고, 군대에 처음 온 신병도 직관적으로 이해할 수 있는 쉬운 표준어와 보충 설명으로 문장을 윤문해 줘.
[안내문 초안]: {Step 2 작성본}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **용어 친화성:** 낯선 군대 은어가 알기 쉬운 표준어로 치환되었는가?
- **원 의미 보존:** 쉬운 말로 바꾸는 과정에서 규정 시간이나 장소의 원래 사실이 왜곡되지 않았는가?

#### 핵심 확인 지표
<blockquote>쉬운 용어로 감수된 신규 전입 병사 첫 주 안내문 최종본

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>신규 전입 병사 첫 주 안내문(필수 산출물)

</li>

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할
스마트폰 모바일 메신저나 병영 앱에서 카드 뉴스로 넘겨볼 수 있는 모바일 최적화 텍스트 포맷으로 추가 변환합니다.

#### 해야 할 일
인쇄해서 벽에 붙일 수 있는 포스터 형태(글자 크게)로도 만들어 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[카드뉴스 텍스트 변환 질의]:** "이 안내문을 부대 모바일 메신저나 앱에서 카드로 넘겨볼 수 있는 '카드뉴스 텍스트 5장' 포맷으로 재구성해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **모바일 가독성:** 한 카드당 3~4줄 이내로 끊어져 작은 화면에서도 가독성이 우수한가?

#### 핵심 확인 지표
<blockquote>모바일 카드뉴스 텍스트 변환본

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>심화 모바일 최적화 메모

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M07] 신규 전입 병사용 안내 만들기 실습 가이드 및 작성 서식

## 1. 신규 전입 병사 첫 주 생활 적응 가이드 (A4 1-Page 요약본)

# 🎖️ [한빛대대] 신병 첫 주 생활 적응 핸드북

> **환영합니다!** 본 가이드는 신규 전입 장병이 자대 배치 첫 주에 꼭 알아야 할 필수 생활 정보와 시설 이용 수칙을 엄선하여 정리한 문서입니다.

---

### 1단계: 전입 당일 (Day 1) - 첫걸음 체크
- **생활관 배정 및 관물대 정리:** 개인 사물함에 보급 피복을 정돈하고 세탁실 위치를 확인합니다.
- **의무실 위치 파악 (본관 1층 서편):** 복용 중인 약이 있거나 진료가 필요할 경우 분대장에게 보고 후 09:00~17:30(소독 12~13:30 제외) 방문합니다.
- **식사 시간 (병영식당):** 아침 07:00 / 점심 12:00 / 저녁 17:30 (배식 10분 전 집합)

### 2단계: 2~3일 차 (Day 2~3) - 편의 시설 활용
- **복지매장(PX, 복지회관 1층):** 평일 12:00~19:30 이용 가능 (세면도구, 위생용품 구입)
- **사이버지식정보방(PC 카페, 복지회관 2층):** 평일 일과 후 17:30~21:00 개방 (부모님 안부 메일 발송)
- **우편물/택배 수령:** 평일 17:30~18:30 본관 당직사령실 옆 보관소에서 수령

### 3단계: 첫 주말 (Day 6~7) - 충전과 여가
- **체력단련실(헬스장, 본관 지하 1층):** 주말 08:00~21:00 자유 체력단련 (운동복 착용 필수)
- **병영도서관(복지회관 2층):** 화~일 13:00~21:00 (1인당 3권, 14일 대출)
- **주말 종교 행사:** 일요일 09:30~12:00 (연병장 우측 종교관)

---

### 🚨 긴급 상황 및 도움 요청 창구
- **몸이 아프거나 응급 환자 발생 시:** 의무실 (내선 103) 또는 당직사령실 (내선 100)
- **병영 생활 고민 및 고충 상담:** 병영생활상담관실 (내선 119 / 핫라인 033-XXX-1119)
- **행정 서류 및 휴가 문의:** 본부 행정반 (내선 101)

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **신병 적응 라이프사이클 타임라인 시트 (Day 1 ~ Day 7):** 전입 당일, 2~3일 차, 첫 주말 시간 순 정보 배치 템플릿
- **1-Page 신병 생활 적응 요약 가이드 서식 (MD):** A4 1장 규격의 헤더, 요약 카드, 체크리스트 마크다운 레이아웃
- **[학습 가이드] 군 행정 용어 순화 사전 및 사용자 맞춤형 문서 작성법:** 신병 눈높이에 맞춘 쉬운 용어 변환 및 가독성 높은 맞춤형 콘텐츠 기획 가이드

**실습 팁:** '사지방', '체단실' 같은 익숙한 줄임말도 신병에게는 낯설 수 있으므로, '사이버지식정보방(PC 카페)'처럼 풀이말을 병기해 주는 것이 좋습니다.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] 대상 맞춤 활용 — 신병 적응 타임라인에 맞춰 생활 정보를 구조화하고 낯선 군대 용어를 알기 쉽게 순화한 A4 1장 안내문을 완성했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>신병 적응 3단계(당일, 2~3일차, 첫 주말) 타임라인 구조화

</li><li>군 행정 용어 및 축약어 순화 감수 완료

</li><li>A4 1장 규격의 가독성 높은 신규 전입 병사 안내문 완성

</li>

#### 채점자 판정 메모
<blockquote>신병 눈높이에서 읽었을 때 이해하기 쉬운지, 전입 당일 꼭 필요한 의무실 위치와 긴급 핫라인이 잘 드러나는지 검토한다.

</blockquote>





In [ ]:
# [미션 07. 신규 전입 병사용 안내 만들기] 학생 실습 프롬프트 및 결과 기록란
mission_07_prompt = """
# [TODO: 미션 07] 신병의 자대 전입 첫 주(Day 1 ~ Day 7) 타임라인별 필수 생활 가이드를 생성하는 프롬프트를 작성하세요.
# 힌트: Day 1(당일), Day 2~3(일과 적응), Day 6~7(첫 주말) 단계별 필수 시설과 긴급 연락처를 포함합니다.
# (여기에 작성한 프롬프트를 입력하세요)
"""

mission_07_result = """
### 📋 [미션 07 실습 결과]
# 🎖️ [한빛대대] 신병 첫 주 생활 적응 핸드북
### 1단계: 전입 당일 (Day 1)
- 
### 2단계: 2~3일 차 (Day 2~3)
- 
### 3단계: 첫 주말 (Day 6~7)
- 
### 🚨 긴급 상황 및 고충 상담 창구
-
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 07. 신규 전입 병사용 안내 만들기", mission_07_prompt, mission_07_result)



---

## 🎯 미션 08. 바뀐 정보 반영하기
**소요시간**: 약 1.5시간

![미션 8 운영 방침 변경 반영 및 지식 베이스 버전 관리 다이어그램](/uploads/common/K09_m08_changelog_sync.png)

### 📋 미션 개요 및 배경
부대 운영 방침이나 시설 이용 시간이 변경되었을 때 AI 도우미가 최신 정보로 즉각 반응하도록 데이터 갱신 및 버전 관리 절차를 수립합니다. 체력단련실 시간 변경 등 가상 시나리오를 바탕으로 안내 자료를 업데이트하고, 변경 전후의 AI 질의 결과를 대조 검증합니다. 안내 문서에 최종 갱신 일자와 변경 이력을 명기하는 데이터 유지보수 프로세스를 훈련합니다.

### 🎯 미션 학습목표
체력단련실 운영 시간 등 변경된 부대 운영 정보를 자료표에 갱신하고, 바뀐 항목에 대해 AI가 최신 내용으로 정확히 답변하는지 검증하며 갱신 일자 표기 규칙을 수립합니다. **이를 통해 규정 변경 시 발생하는 정보 불일치 문제를 해결하고, 지식 베이스를 지속적으로 최신화·유지보수하는 데이터 수명주기 관리 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 변경 안내 2건을 자료표에 반영한다.
#### 이 단계의 역할
실습자료에 하달된 '체력단련실 운영 시간 단축' 및 '의무실 물리치료실 예약제 전환' 2건의 변경 사항을 원본 자료표에 정확히 업데이트합니다.

#### 해야 할 일

<li>가상 변경 공지문 2건의 변경 전 수치와 변경 후 수치를 확인합니다.

</li><li>미션2에서 만든 자료표의 해당 시설 행을 찾아 새로운 시간과 절차로 수정합니다.

</li><li>AI에게 변경된 데이터표를 검토하게 하여 기존 항목과의 충돌이나 오타를 검증합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[시설 운영 변경안 데이터 갱신 질의]:** "아래 [변경 공지 2건]을 확인하고, 기존 [부대 안내 자료표]에서 체력단련실과 의무실 정보를 정확히 수정해 줘. 변경된 부분은 굵은 글씨(**)로 표시해 줘.
[변경 공지]: 1. 체력단련실 평일 마감 21:00->20:00 단축 / 2. 의무실 물리치료 매주 화,목 예약제 신설
[기존 자료표]: {미션2 자료표}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **정확한 핀포인트 수정:** 변경된 시간과 조건만 정확히 수정되고 나머지 10개 시설은 온전히 유지되었는가?
- **오타 방지:** 수정 과정에서 기존의 연락처나 위치 정보가 훼손되지 않았는가?

#### 핵심 확인 지표
<blockquote>변경 사항 2건이 반영된 최신 부대 안내 자료표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>갱신된 안내 자료표(필수 산출물)

</li>

#### 🔹 Step 2. 바뀐 항목을 다시 질문한다.
#### 이 단계의 역할
갱신된 자료표를 주입한 프롬프트에 변경된 시설에 대한 질문을 재질의하여 AI가 옛날 정보를 버리고 새 정보로 답하는지 확인합니다.

#### 해야 할 일

<li>갱신된 최신 자료표를 프롬프트에 장착합니다.

</li><li>"체단실 평일에 몇 시까지 운동할 수 있어?", "물리치료 받으려면 어떻게 해?"를 질의합니다.

</li><li>구버전 데이터 주입 시의 답변과 신버전 데이터 주입 시의 답변을 나란히 비교표로 정리합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[데이터 갱신 검증 질의]:** "[최신 자료표 주입]: {갱신된 자료표}
[질문 1]: 체력단련실 평일 운영 시간 마감이 몇 시인가요?
[질문 2]: 의무실에서 물리치료를 받고 싶은데 언제 가야 하나요?"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **구 정보 잔존 차단:** 과거의 마감 시간(21:00)을 말하지 않고 새로운 마감 시간(20:00)을 정확히 인용하는가?
- **예약제 조건 인지:** 물리치료실 화/목 예약제 규정을 답변에 포함하였는가?

#### 핵심 확인 지표
<blockquote>갱신 전후 질의응답 비교 대조표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>갱신 전후 답변 비교(필수 산출물)

</li>

#### 🔹 Step 3. 갱신 날짜 규칙을 만든다.
#### 이 단계의 역할
정보 노후화 및 버전 혼선을 방지하기 위해 안내 자료표 상단에 최종 갱신 일자(Last-Updated)와 버전 번호, 변경 이력을 기록하는 관리 규칙을 수립합니다.

#### 해야 할 일

<li>표 상단에 [문서 버전: v1.1 | 최종 수정일: 2026-XX-XX | 수정자: 행정병] 헤더 서식을 규정합니다.

</li><li>자료 변경 시 변경 사유와 변경 항목을 하단에 [변경 이력(Changelog)]으로 누적 기록하는 절차를 명문화합니다.

</li><li>AI 프롬프트 답변 끝에도 "이 안내는 YYYY.MM.DD 기준 최신 정보입니다"를 명시하도록 규칙을 추가합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[데이터 버전 관리 규칙 정립 질의]:** "부대 안내 자료표가 지속적으로 개정될 때 버전 혼선을 방지할 수 있는 '문서 버전 관리 및 갱신 표기 지침'을 3단계 수칙으로 정립해 줘. 그리고 AI 답변 끝에 최신 갱신 일자를 자동으로 표기하는 프롬프트 규칙 문장을 만들어 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **규칙 구체성:** 버전 넘버링 기준과 수정 일자 표기 위치가 명확히 규정되었는가?
- **답변 자동 반영:** AI가 답변 말미에 최신 기준 일자를 출력할 수 있도록 설계되었는가?

#### 핵심 확인 지표
<blockquote>부대 안내 자료 버전 관리 및 갱신 규칙서

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>갱신 규칙(필수 산출물)

</li>

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할
변경 예정 공지(D-Day 예고)가 있을 때 현재 기준과 향후 적용 기준을 구분하여 안내하는 시차 반영 프롬프팅을 테스트합니다.

#### 해야 할 일
'예전 정보로 답했을 때 사용자가 지적하면 다시 확인하는 대화'를 작성해 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[시차 변경 공지 처리 질의]:** "[시차 안내 지침]: 현재 적용 규정과 향후 변경 예정 규정이 함께 있을 경우, '현재는 OO이나, MM월 DD일부터 XX로 변경 예정'임을 병기할 것."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **시점 구분:** 현재 유효한 정보와 미래 변경 예정 사항이 명확히 분리되는가?

#### 핵심 확인 지표
<blockquote>시차 발생 공지 처리 시험 메모

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>심화 변경 관리 보고서

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M08] 바뀐 정보 반영하기 실습 가이드 및 작성 서식

## 1. 갱신된 부대 생활 안내 자료표 (v1.1)

| 연번 | 시설명 | 위치 | 갱신된 운영 시간 | 예외 운영 조건 | 문의처 | 갱신된 제공 업무 |
| :---: | :--- | :--- | :--- | :--- | :--- | :--- |
| 1 | 대대 의무실 | 본관 1층 서편 | 평일 09:00~17:30 | 12:00~13:30 소독, 야간 당직 접수 | 내선 103 | 일반 진료, 약 조제, **물리치료(화·목 예약제)** |
| 3 | 체력단련실 | 본관 지하 1층 | **평일 06:30~20:00 (1시간 단축)**, 주말 08:00~21:00 | 일과 중 개인 이용 불가, 수 14~16시 방역 | 내선 104 | 웨이트 트레이닝, 유산소, 샤워장 |
*(이외 10개 시설 정보는 v1.0과 동일 유지)*

---

## 2. 갱신 전후 질의응답 비교표

| 질문 내용 | 갱신 전 답변 (v1.0 데이터) | 갱신 후 답변 (v1.1 데이터) | 갱신 검증 판정 |
| :--- | :--- | :--- | :---: |
| "체단실 평일 마감 몇 시야?" | "평일 21:00까지 운영합니다." | "📌 요약 답변: 평일 20:00까지 단축 운영합니다." | **성공 (최신화 완료)** |
| "물리치료 당일 바로 가도 돼?"| "평일 진료 시간 내 방문 가능합니다." | "📌 요약 답변: 물리치료는 매주 화·목 예약제입니다." | **성공 (신규 규정 인용)** |

---

## 3. 부대 안내 데이터 버전 관리 규정 (Changelog Rule)
1. **문서 헤더 표기:** 모든 자료표 최상단에 `[버전: vX.X | 최종 갱신일: YYYY.MM.DD | 승인관: OO과장]` 명기
2. **변경 이력 관리:** 변경 발생 시 일자, 변경 시설, 변경 내용, 하달 공문 번호를 표 하단에 누적 기록
3. **AI 프롬프트 연동:** 프롬프트 시스템 지시문에 `[기준 일자: YYYY.MM.DD 현재]`를 명시하여 AI가 답변 시 최신 기준임을 보증

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **가상 부대 운영 방침 변경 공지 2건 (MD):** 체력단련실 단축 운영 및 의무실 물리치료 예약제 시행 공문
- **데이터 버전 관리 및 변경 이력(Changelog) 양식:** 문서 버전(v1.1), 갱신 일자, 변경 사유 누적 기록 서식
- **[학습 가이드] 지식 베이스(Knowledge Base) 수명주기 관리 및 데이터 최신화:** 규정 변경 시 발생하는 정보 불일치 방지 및 RAG 데이터 갱신 프로세스

**실습 팁:** 자료를 바꾼 뒤에는 반드시 과거 정보로 질문을 던져, AI가 이전의 잘못된 운영 시간을 답하지 않는지 '과거 잔존 데이터 테스트'를 진행하세요.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] 자료 갱신·검토 — 운영 방침 변경 사항 2건을 정확히 갱신하고 버전 관리 규칙 수립 및 최신 답변 출력을 검증했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>변경 공지 2건이 반영된 v1.1 갱신 안내 자료표 완성

</li><li>갱신 전후 질의응답 비교 대조를 통한 최신성 검증

</li><li>문서 버전 표기 및 변경 이력(Changelog) 관리 규칙 수립

</li>

#### 채점자 판정 메모
<blockquote>체단실 평일 마감 시간이 20:00으로 정확히 답변되는지, 물리치료 예약제 조건이 빠짐없이 안내되는지 확인한다.

</blockquote>





In [ ]:
# [미션 08. 바뀐 정보 반영하기] 학생 실습 프롬프트 및 결과 기록란
mission_08_prompt = """
# [TODO: 미션 08] 부대 시설 운영 변경 공지 2건을 반영하여 자료표를 갱신(v1.1)하고 버전 관리 이력을 작성하는 프롬프트를 작성하세요.
# 힌트: load_dataset(8)의 updates를 확인하고, 체단실 시간 단축과 물리치료실 예약제 변경을 반영합니다.
# (여기에 작성한 프롬프트를 입력하세요)
"""

mission_08_result = """
### 📋 [미션 08 실습 결과]
## 1. 갱신된 부대 생활 안내 자료표 (v1.1 변경 사항)
| 연번 | 시설명 | 갱신된 운영 시간 | 갱신된 제공 업무 | 문의처 |
| :---: | :--- | :--- | :--- | :--- |
| **1** | 대대 의무실 | | | |
| **3** | 체력단련실 | | | |

## 2. 버전 관리 이력 (Changelog)
- 최종 갱신 일자: 
- 변경 사유:
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 08. 바뀐 정보 반영하기", mission_08_prompt, mission_08_result)



---

## 🎯 미션 09. 나만의 부대 생활 질문 목록 만들기
**소요시간**: 약 1.5시간

![미션 9 10개 벤치마크 평가 질의 세트 및 정량 정확도 산출 다이어그램](/uploads/common/K09_m09_benchmark_eval.png)

### 📋 미션 개요 및 배경
완성 단계의 도우미 성능을 객관적으로 측정하기 위해 정상 질문 7개와 자료 밖 예외 질문 3개로 구성된 테스트 질의 세트(Benchmark)를 설계합니다. 사전에 정의된 모범 정답과 AI의 답변을 비교하여 정확도 지표를 정량적으로 산출하고, 오답 패턴을 분석해 최종 프롬프트 파라미터와 지시문을 튜닝합니다. AI 시스템의 품질을 스스로 측정하고 보완하는 테스트 주도 개선 역량을 기릅니다.

### 🎯 미션 학습목표
정상 질의 7개와 자료 밖 예외 질의 3개로 구성된 10개의 테스트 질문 세트를 설계하고, AI 답변의 정답률을 정량적으로 산출하여 미흡한 프롬프트를 반복 교정합니다. **이를 통해 AI 시스템의 성능을 정량 평가하는 벤치마크 테스트 방법론과 오류 분석 기반의 반복 개선(Iterative Refinement) 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 시험 질문 10개(정상 7개, 자료 밖 3개)를 선정한다.
#### 이 단계의 역할
도우미의 종합 성능을 체계적으로 측정할 수 있는 10개의 벤치마크 테스트 질문 세트(자료 내 정상 질의 7건, 자료 밖 예외 질의 3건)를 구성합니다.

#### 해야 할 일

<li>정상 질문 7건(시간, 위치, 문의처, 예외 조건 등 다양한 카테고리 골고루 분배)을 설계합니다.

</li><li>자료 밖 질문 3건(포상휴가 기준, 두발 규정, 사격 훈련 일정 등 부대 내 실제 자주 묻지만 자료에는 없는 항목)을 구성합니다.

</li><li>AI를 활용하여 질문의 모호성을 검토하고 10개 질문의 대표성을 확보합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[벤치마크 테스트 질문 세트 설계 질의]:** "부대 안내 도우미 성능을 평가하기 위해, 제공된 자료표에서 답변할 수 있는 다양한 유형의 정상 질문 7개와, 자료에는 없지만 신병들이 실제로 자주 묻는 자료 밖 질문 3개를 만들어 줘.
[자료표]: {갱신된 자료표}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **질문 비율 준수:** 정상 질문 7건과 자료 밖 질문 3건의 7:3 비율이 정확히 준수되었는가?
- **질문 다양성:** 단순 위치 질문 외에 소독 시간이나 예약제 등 조건부 질문이 포함되었는가?

#### 핵심 확인 지표
<blockquote>10개 벤치마크 시험 질문 세트

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>시험 질문 10개 목록

</li>

#### 🔹 Step 2. 시험 질문 10개에 대한 모범 정답표를 작성한다.
#### 이 단계의 역할
AI 답변의 채점 기준이 될 10개 질문의 정답 기준(Ground Truth)을 사전에 명확히 작성합니다.

#### 해야 할 일

<li>정상 질문 7건에 대해 자료표 원문과 100% 일치하는 모범 정답(시간, 위치, 근거)을 기술합니다.

</li><li>자료 밖 질문 3건에 대해 '자료 미등록 안내 및 행정반/당직실 문의' 표준 거절 응답을 정답으로 명시합니다.

</li><li>채점관 관점에서 핵심 키워드 채점 기준(채점 루브릭)을 정리합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[모범 정답표(Ground Truth) 작성 질의]:** "위 10개 시험 질문 각각에 대해, 안내 자료표와 예외 처리 원칙에 근거한 무결점 모범 정답(Ground Truth)과 필수 포함 핵심 키워드를 표 형태로 작성해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **정답 무결성:** 정상 질문 정답이 최신 자료표와 한 글자도 어긋남이 없는가?
- **예외 정답 일관성:** 자료 밖 질문 3건의 정답이 모두 표준 Fallback 문구로 지정되었는가?

#### 핵심 확인 지표
<blockquote>10개 시험 질문 모범 정답표 및 채점 루브릭

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>시험 질문 10개와 정답(필수 산출물)

</li>

#### 🔹 Step 3. 도우미 답과 모범 정답을 비교해 정확도를 계산한다.
#### 이 단계의 역할
현재 구축된 도우미 프롬프트에 10개 시험 질문을 일괄 입력하고, 모범 정답과 비교하여 정량적 정확도(Score %)를 산출합니다.

#### 해야 할 일

<li>완성된 프롬프트에 10개 질문을 순차적으로 입력하고 응답을 기록합니다.

</li><li>각 문항별로 정답(1점), 부분정답(0.5점), 오답(0점)으로 채점합니다.

</li><li>총점과 정확도 백분율(예: 8.5/10 = 85%)을 계산하고 오답 문항을 특정합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[벤치마크 자동 채점 및 정확도 산출 질의]:** "아래 [AI 응답 10건]을 앞서 작성한 [모범 정답 10건]과 1:1로 채점해 줘. 각 문항별 일치 여부(PASS/FAIL)와 감점 사유, 최종 정확도(%)를 계산해 줘.
[모범 정답]: {Step 2 정답표}
[AI 응답]: {수집된 응답}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **채점 객관성:** 서식 누락, 시간 오차, 불완전 거절 등을 엄격히 감점 처리했는가?
- **정확도 산출:** 10개 문항에 대한 정량 지표(%)가 수학적으로 정확히 계산되었는가?

#### 핵심 확인 지표
<blockquote>10개 문항 채점 결과표 및 정확도(%) 산출 기록

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>정확도 기록(필수 산출물)

</li>

#### 🔹 Step 4. 틀린 질문을 분석하여 프롬프트를 개선한다.
#### 이 단계의 역할
감점이나 오답이 발생한 문항의 취약점을 분석하고, 이를 극복할 수 있도록 프롬프트 지시문과 제약조건을 튜닝합니다.

#### 해야 할 일

<li>오답이 발생한 문항(조건부 시간 누락, 질문의 의도 오독 등)의 근본 원인을 파악합니다.

</li><li>프롬프트 내 지시사항 보강, 예시(Few-shot) 추가, 금지 조항 구체화 등 수정 작업을 수행합니다.

</li><li>개선된 프롬프트로 틀렸던 질문을 재시험하여 정답(정확도 향상)으로 개선되었는지 재검증합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[오답 분석 기반 프롬프트 튜닝 질의]:** "질문 4번에서 AI가 점심 소독 시간 안내를 누락하여 감점되었어. 프롬프트의 지시사항 중 어떤 문구를 수정해야 향후 소독 시간 같은 세부 조건을 절대 누락하지 않고 완벽히 출력할 수 있을지 개선안을 제시해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **재시험 통과:** 프롬프트 수정 후 취약 문항이 완벽하게 정답으로 개선되었는가?
- **회귀 오류 방지:** 수정 과정에서 다른 정상 문항의 답변 품질이 저하되지 않았는가?

#### 핵심 확인 지표
<blockquote>취약점 개선 전후 비교표 및 최종 튜닝 프롬프트

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>수정 프롬프트(필수 산출물)

</li>

#### 🔹 Step 5. 심화 도전 과제 및 최적화
#### 이 단계의 역할
동료 장병(또는 가상 짝)의 도우미 프롬프트와 내 시험 질문 세트를 상호 교환(Cross-testing)하여 시스템의 일반화 성능을 검증합니다.

#### 해야 할 일
짝의 도우미에 내 시험 질문을 넣어 비교해 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[교차 검증 질의]:** "다른 사용자가 만든 다음 5개의 돌발 질문을 내 도우미 프롬프트에 넣었을 때의 견고성(Robustness)을 평가해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **미지의 질문 방어:** 다른 사람의 독특한 질문 스타일에도 5단 서식과 정확도가 유지되는가?

#### 핵심 확인 지표
<blockquote>상호 교차 테스트 평가 메모

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>심화 교차 검증 메모

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M09] 나만의 부대 생활 질문 목록 만들기 실습 가이드 및 작성 서식

## 1. 10개 벤치마크 평가 질의 및 모범 정답표 (Ground Truth)

| 번호 | 유형 | 시험 질문 내용 | 모범 정답 요약 (Ground Truth) | 필수 포함 키워드 |
| :---: | :---: | :--- | :--- | :--- |
| **01** | 정상 | 의무실 점심 소독 시간에 진료 가능한가? | 12:00~13:30 소독으로 일반 진료 불가 | 12:00~13:30, 소독, 진료 불가 |
| **02** | 정상 | 체단실 평일 마감 시간은 몇 시인가? | 평일 20:00 마감 (단축 운영 반영) | 평일 20:00 |
| **03** | 정상 | 충성마트(PX) 일요일 이용 가능한가? | 일요일 정기 휴무로 이용 불가 | 일요일, 휴무 |
| **04** | 정상 | 병영도서관 대출 권수와 기한은? | 1인 3권, 14일간 대출 가능 | 3권, 14일 |
| **05** | 정상 | 이발실 바로 가면 이용 가능한가? | 화·목·토 사전 예약제, 즉시 방문 불가 | 사전 예약제, 선착순 불가 |
| **06** | 정상 | 사제 택배 당일 수령 가능한가? | 보안검색 후 익일 17:30~18:30 수령 | 익일, 17:30~18:30 |
| **07** | 정상 | 토요일 아침 식사 시간은 언제인가? | 브런치데이 운영으로 09:00~10:30 | 브런치, 09:00~10:30 |
| **08** | 예외 | 정기휴가 심사 발표일은 언제인가? | 안내 자료 미등록 정보 (행정반 문의) | 자료 미등록, 행정반 문의 |
| **09** | 예외 | 외박 신청에 필요한 상점 점수는? | 안내 자료 미등록 정보 (행정반 문의) | 자료 미등록, 행정반 문의 |
| **10** | 예외 | 다음 달 유격 훈련 장소는 어디인가? | 안내 자료 미등록 정보 (행정반 문의) | 자료 미등록, 행정반 문의 |

---

## 2. 벤치마크 채점표 및 최종 정확도 기록
- **정상 질문 정답률:** 7 / 7 (100%)
- **자료 밖 질문 차단율:** 3 / 3 (100%)
- **서식(5단 템플릿) 준수율:** 10 / 10 (100%)
- **종합 정확도 (Score):** **100% (10 / 10 PASS)**

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **10대 벤치마크 평가 질의 세트 양식 (정상 7건 : 예외 3건):** 대표성 있는 질의 문항 설계 시트
- **모범 정답표(Ground Truth) 및 채점 루브릭 (Scoring Rubric):** 정답, 부분정답, 오답 채점 기준 및 핵심 키워드 체크리스트
- **[학습 가이드] LLM 정량 평가(Evaluation) 방법론 및 반복 프롬프트 튜닝:** 정답률(Accuracy %) 산출과 오류 패턴 분석을 통한 프롬프트 개선 사이클

**실습 팁:** 10개 질문 중 정상 질문 7개는 시간, 장소, 절차 등 시설을 고루 섞고, 예외 질문 3개는 신병들이 진짜 궁금해할 법한 규정 외 질문(월급, 훈련 등)으로 구성하세요.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] 나만의 활용법(시험·개선) — 정상 질의와 자료 밖 질의로 10개 벤치마크를 구축하고 정답률 산출 및 오답 원인을 프롬프트에 개선했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>10개 벤치마크 평가 질의 및 모범 정답표(Ground Truth) 작성 완료

</li><li>10개 문항 채점 결과표 및 정량 정확도(%) 산출 기록

</li><li>취약점 보완을 위한 프롬프트 수정 전후 비교표 및 최종 튜닝 프롬프트 도출

</li>

#### 채점자 판정 메모
<blockquote>정답률 수치 자체보다, 채점 결과 틀린 문항의 원인을 정확히 짚어내고 프롬프트를 고쳐서 개선해 냈는지를 중점 평가한다.

</blockquote>





In [ ]:
# [미션 09. 나만의 부대 생활 질문 목록 만들기] 학생 실습 프롬프트 및 결과 기록란
mission_09_prompt = """
# [TODO: 미션 09] 정상 질문 7건과 자료 밖 예외 질문 3건으로 구성된 10대 벤치마크 세트를 설계하고 정확도를 평가하는 프롬프트를 작성하세요.
# 힌트: load_dataset(9)의 benchmarkSet을 확인하고, Ground Truth 대조 채점표 및 최종 정답률 산출 방식을 구성합니다.
# (여기에 작성한 프롬프트를 입력하세요)
"""

mission_09_result = """
### 📋 [미션 09 실습 결과]
## 1. 10개 벤치마크 채점표
| 번호 | 유형 | 시험 질문 내용 | 도우미 답변 평가 | 채점 (PASS/FAIL) |
| :---: | :---: | :--- | :--- | :---: |
| **01** | 정상 | 의무실 소독 시간 진료 여부 | | |
| **02** | 정상 | 사지방 화요일 점검 시간 | | |
| **03** | 정상 | PX 토요일 운영 시간 | | |
| **04** | 정상 | 체단실 수요일 방역 시간 | | |
| **05** | 정상 | 도서관 대출 권수 및 반납 기한 | | |
| **06** | 정상 | 세탁기 고장 신고처 | | |
| **07** | 정상 | 이발실 휴무일 및 예약 | | |
| **08** | 예외 | 당직 사령 근무 수당 | | |
| **09** | 예외 | 주말 외출/외박 가능 시간 | | |
| **10** | 예외 | 영내 노래방 코인 가격 | | |

- **최종 정확도 산출**: (   ) / 10 = (   )%
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 09. 나만의 부대 생활 질문 목록 만들기", mission_09_prompt, mission_09_result)



---

## 🎯 미션 10. 부대 생활 정보 안내 도우미 완성하기
**소요시간**: 약 1.5시간

![미션 10 부대 생활 정보 안내 도우미 통합 운영 매뉴얼 패키징 다이어그램](/uploads/common/K09_m10_system_manual.png)

### 📋 미션 개요 및 배경
그동안 단계별로 제작·검증한 부대 안내 자료표, 기반 프롬프트, 예외 처리 규칙, 출력 서식을 통합하여 최종 '부대 생활 정보 안내 도우미 사용 설명서'를 편찬합니다. 성능 평가 결과와 함께 'AI 답변은 참고용이며 최종 확인은 담당 부서 지침을 따른다'는 인간 책임 원칙(HITL) 안내 문구를 반드시 포함합니다. 실제 부대 실무 현장에서 즉시 활용 가능한 신뢰도 높은 업무 지원 시스템 패키지를 완성합니다.

### 🎯 미션 학습목표
안내 자료표, 질의응답 프롬프트, 예외 대응 규칙, 출력 템플릿 및 HITL(인간 책임) 원칙을 집약한 최종 사용 설명서를 완성하고 종합 검증을 수행합니다. **이를 통해 단위 실습 결과물들을 현장 배포 가능한 완성형 업무 시스템으로 패키징하고, 책임 있는 AI 활용 원칙을 종합적으로 실천하는 완결 역량을 체득합니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 자료표·프롬프트·예외 규칙·형식을 사용 설명서로 묶는다.
#### 이 단계의 역할
앞선 미션들에서 개별 제작하고 검증한 4대 핵심 컴포넌트(자료표, 기본 프롬프트, Fallback 규칙, 5단 출력 서식)를 하나의 일체형 운영 매뉴얼로 집대성합니다.

#### 해야 할 일

<li>'부대 생활 정보 안내 도우미 사용 설명서'의 목차 체계(개요, 데이터베이스, 시스템 프롬프트, 운영 수칙)를 구성합니다.

</li><li>최신 부대 안내 자료표, Few-shot 프롬프트, 예외 처리 규칙, 출력 템플릿을 한 문서 안에 체계적으로 병합합니다.

</li><li>AI에게 문서 전체의 어조 일관성과 마크다운 포맷 정렬을 검토받습니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[통합 운영 매뉴얼 편찬 질의]:** "우리가 지금까지 완성한 [최신 안내 자료표], [자료 기반 프롬프트], [예외 처리 규칙], [5단 출력 포맷] 4개 요소를 하나의 완성형 '부대 생활 안내 AI 도우미 운영 매뉴얼(사용 설명서)' 마크다운 문서로 통합 편찬해 줘.
[자료표]: {미션8 자료표}
[프롬프트]: {미션9 수정 프롬프트}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **4대 요소 완전 결합:** 자료표, 프롬프트, 예외 규칙, 서식이 누락 없이 일체형으로 결합되었는가?
- **실무 가독성:** 다른 장병이 바로 복사해서 실행할 수 있을 만큼 설명서 구조가 직관적인가?

#### 핵심 확인 지표
<blockquote>통합 부대 생활 안내 도우미 사용 설명서 초안

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>사용 설명서 통합 초안

</li>

#### 🔹 Step 2. 시험 결과를 넣는다.
#### 이 단계의 역할
사용 설명서를 이용할 간부나 후임병이 도우미의 신뢰도를 한눈에 확인할 수 있도록 미션9의 정량적 벤치마크 테스트 결과(정확도 지표 및 성능 검증표)를 문서에 수록합니다.

#### 해야 할 일

<li>10개 시험 질문에 대한 최종 정확도(예: 100% 또는 95%) 지표를 요약 카드로 작성합니다.

</li><li>정상 질의 처리율, 자료 밖 질문 차단율, 서식 준수율 3대 세부 지표를 표 형태로 삽입합니다.

</li><li>성능 측정 일자와 테스트 환경을 문서에 명시합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[성능 검증 보고서 요약 질의]:** "통합 매뉴얼의 '시스템 성능 및 신뢰도 검증' 섹션에 들어갈 [10개 벤치마크 테스트 결과 요약표]를 작성해 줘. (정상 질문 처리율 100%, 미등록 질문 차단율 100%, 서식 준수율 100% 명시)"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **지표 신뢰성:** 미션9에서 실제로 도출된 정량 데이터와 완벽히 부합하는가?
- **검증 객관성:** 평가 기준과 테스트 환경이 명확히 제시되었는가?

#### 핵심 확인 지표
<blockquote>정량 성능 평가표가 포함된 사용 설명서 중간본

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>시험 결과 포함 중간본

</li>

#### 🔹 Step 3. 확인 안내(인간 책임 원칙, HITL)를 넣는다.
#### 이 단계의 역할
AI는 어디까지나 보조 도구이며, 최종 확인과 책임은 인간(담당 부서 및 지휘계통)에 있음을 명시하는 안전 면책 및 HITL 수칙을 설명서에 명기합니다.

#### 해야 할 일

<li>"본 도우미의 답변은 편의를 위한 참고 자료이며, 실제 일정, 규정, 진료 예약 등의 최종 확인은 반드시 행정반 및 담당 부서의 공식 지침을 따라야 한다"는 경고문을 작성합니다.

</li><li>비인가 군사 비밀이나 전우의 개인정보 입력을 엄격히 금지하는 군 보안 준수 서약을 포함합니다.

</li><li>시스템 관리자(행정병)의 문의처 및 오류 신고 절차를 기재합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[인간 책임(HITL) 및 군사 보안 안내문 질의]:** "군 부대 실무 현장에서 생성형 AI 도우미를 배포할 때 필수적인 '인간 책임 원칙(HITL) 안내문' 및 '군사 보안 준수 경고문'을 가장 엄격하고 신뢰감 있는 표준 문안으로 작성해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **인간 책임 원칙 명시:** AI 답변의 보조적 성격과 최종 공식 확인 창구(행정반)가 뚜렷이 명기되었는가?
- **군사 보안 경고:** 실제 군사 비밀 및 개인정보 입력 금지 수칙이 강력하게 고지되었는가?

#### 핵심 확인 지표
<blockquote>HITL 인간 책임 원칙 경고문 및 보안 서약 문안

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>확인 안내 문구가 포함된 사용 설명서

</li>

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할
차기 운영자(후임 행정병)를 위한 '도우미 유지보수 및 트러블슈팅(Troubleshooting) Q&A 가이드'를 매뉴얼 부록으로 추가하여 최종본을 완성합니다.

#### 해야 할 일
새로 들어온 후임이 혼자서도 도우미를 쓸 수 있는지 사용 설명서를 읽게 하고 피드백을 받아 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[운영자 트러블슈팅 가이드 질의]:** "후임 행정병이 인수인계받아 이 도우미를 계속 관리할 수 있도록 '운영자 트러블슈팅 가이드 3문 3답'을 부록으로 작성해 줘. (새 시설 추가 방법, 환각 발생 시 대처법, 프롬프트 백업 수칙)"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **인수인계 실용성:** AI 비전공자도 따라 할 수 있을 만큼 유지보수 절차가 명확한가?
- **최종 완성도:** 부록까지 완비되어 즉시 실무에 배포 가능한 수준인가?

#### 핵심 확인 지표
<blockquote>부록(트러블슈팅)까지 완비된 최종 완성본

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>부대 생활 정보 안내 도우미 사용 설명서(최종본) (필수)

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M10] 부대 생활 정보 안내 도우미 운영 매뉴얼 및 사용 설명서 (최종본)

# 📘 한빛대대 부대 생활 정보 안내 도우미 운영 매뉴얼

---

## 1. 시스템 개요 및 목적
- **목적:** 신규 전입 장병의 부대 적응을 돕고, 일상적인 생활 안내 문의를 3초 내에 정확히 해결하여 행정 소요를 절감함.
- **핵심 원리:** 외부 AI의 지식을 차단하고, 오직 검증된 12개 부대 시설 안내 데이터만을 근거로 응답하는 검색 증강(RAG) 프롬프트 구조.

---

## 2. 통합 시스템 프롬프트 전문 (Copy & Paste 사용 가능)

```markdown
[시스템 역할]: 너는 육군 가상 한빛대대의 공식 부대 생활 안내 도우미 '한빛봇'이다.
[대원칙]:
1. 반드시 아래 [공식 부대 안내 자료]에 등록된 내용에만 100% 근거하여 사실대로 답변하라.
2. 자료에 명시되지 않은 질문에는 일체 지어내지 말고 "⚠️ 안내 자료 미등록 정보: 행정반(내선 101) 문의"로 응답하라.
3. 사용자 질문의 어투(반말, 은어, 축약어)와 무관하게 항상 아래 [5단 고정 서식]을 유지하라.

[5단 고정 서식]:
📌 요약 답변: [한 문장 결론]
📍 시설 위치: [건물 및 층수]
⏰ 운영 시간: [정규 시간 및 예외 조건]
📞 문의 연락: [담당 부서/내선]
🔍 인용 근거: [참조 시설명]

[질의응답 모범 예시]:
질문: 의무실 진료 언제까지 해?
답변:
📌 요약 답변: 평일 17:30까지 일반 진료를 받으실 수 있습니다.
📍 시설 위치: 본관 1층 서편 의무실
⏰ 운영 시간: 평일 09:00~17:30 (점심 소독 12:00~13:30 제외, 야간은 당직 접수)
📞 문의 연락: 내선 103 (의무대)
🔍 인용 근거: [대대 의무실 운영 안내]

[공식 부대 안내 자료 v1.1]:
{미션8 갱신 자료표 전문 삽입}
```

---

## 3. 벤치마크 테스트 검증 결과
- **테스트 일자:** 2026년 11월
- **테스트 문항:** 총 10문항 (정상 7건, 자료 밖 3건)
- **달성 성과:** 정확도 100%, 환각(거짓 정보) 발생률 0%, 서식 준수율 100%

---

## 4. ⚠️ 인간 책임 원칙 (HITL) 및 군사 보안 수칙 (필독)
1. **AI 답변은 보조 참고용:** AI가 생성한 답변은 편의를 위한 1차 안내 자료이며, 실제 일정, 규정 해석, 진료 예약 등의 최종 효력은 담당 부서(행정반)의 공식 지침을 따릅니다.
2. **군사 보안 준수:** 본 도우미에는 교육용 가상 데이터만 주입되어 있습니다. 실제 부대 비밀, 훈련 계획, 간부 개인 연락처를 외부 AI에 절대 입력하지 마십시오.
3. **담당 관리자:** 본부포대 행정병 (내선 101)

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **최종 '부대 생활 정보 안내 도우미 사용 설명서' 종합 서식 (MD):** 데이터표, 최종 프롬프트, 운영 수칙 통합 템플릿
- **시스템 성능 검증 보고서 요약 카드 양식:** 10개 벤치마크 테스트 결과 및 지표 요약표
- **[학습 가이드] 책임 있는 AI(Responsible AI)와 배포 패키징 가이드:** 인간 책임 원칙(HITL) 안내문, 군사 보안 준수 경고문, 후임병 인수인계 트러블슈팅 매뉴얼

**실습 팁:** 설명서 맨 앞과 끝에는 반드시 "본 도우미는 참고용이며 최종 승인은 지휘계통을 따른다"는 HITL 문구를 굵은 글씨로 배치하여 배포해야 합니다.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] 결과물 완성 — 4대 컴포넌트를 통합한 사용 설명서를 편찬하고 인간 책임 원칙(HITL) 및 군사 보안 경고문을 포함했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>부대 생활 정보 안내 도우미 사용 설명서 최종본 완성

</li><li>4대 핵심 구성 요소(데이터표, 프롬프트, 예외규칙, 서식)의 완전한 결합

</li><li>정량 성능 평가 지표 및 인간 책임 원칙(HITL), 군사 보안 경고문 수록

</li>

#### 채점자 판정 메모
<blockquote>실제 부대 행정반에서 후임병이나 동료 장병이 이 설명서만 보고도 도우미를 즉시 운영할 수 있는지 실용성과 안전성을 종합 평가한다.

</blockquote>





In [ ]:
# [미션 10. 부대 생활 정보 안내 도우미 완성하기] 학생 실습 프롬프트 및 결과 기록란
mission_10_prompt = """
# [TODO: 미션 10] 시스템 개요, 통합 프롬프트, 벤치마크 검증 결과, HITL(인간 책임) 원칙을 포함하는 최종 운영 매뉴얼을 작성하세요.
# 힌트: 미션 1~9의 성과물을 집약하여 실무 현장에서 즉시 사용할 수 있는 표준 매뉴얼 양식으로 패키징합니다.
# (여기에 작성한 프롬프트를 입력하세요)
"""

mission_10_result = """
### 📋 [미션 10 실습 결과]
# 📘 한빛대대 부대 생활 정보 안내 도우미 운영 매뉴얼
## 1. 시스템 개요 및 목적
## 2. 통합 시스템 프롬프트 전문
## 3. 벤치마크 테스트 성능 검증 결과
## 4. ⚠️ 인간 책임 원칙 (HITL) 및 군사 보안 수칙
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 10. 부대 생활 정보 안내 도우미 완성하기", mission_10_prompt, mission_10_result)

